In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Installation
# ══════════════════════════════════════════════════════════════════════════════
import subprocess

def _run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode == 0 else '❌'} {label or cmd[:60]}")
    if r.returncode != 0:
        print(r.stderr[-300:])
    return r.returncode == 0

_run("apt-get install -y -q tesseract-ocr tesseract-ocr-ara poppler-utils",
     "Tesseract OCR arabe + Poppler")
_run("pip install -q pytesseract Pillow pymupdf",           "pytesseract + Pillow + PyMuPDF")
_run("pip install -q transformers sentence-transformers",   "Sentence-Transformers")
_run("pip install -q torch --index-url https://download.pytorch.org/whl/cu118", "PyTorch CUDA")
_run("pip install -q rank_bm25",                            "BM25")

if not _run("pip install -q faiss-gpu-cu11", "FAISS GPU"):
    _run("pip install -q faiss-cpu", "FAISS CPU (fallback)")

_run("pip install -q llama-cpp-python "
     "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122",
     "llama-cpp CUDA")
_run("pip install -q huggingface_hub tqdm ipywidgets", "HuggingFace Hub + widgets")
print("\n Installation complète.")


✅ Tesseract OCR arabe + Poppler
✅ pytesseract + Pillow + PyMuPDF
✅ Sentence-Transformers
✅ PyTorch CUDA
✅ BM25
✅ FAISS GPU
✅ llama-cpp CUDA
✅ HuggingFace Hub + widgets

 Installation complète.


In [11]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — OCR brut uniquement (v6 — optimisée vitesse)
#
# OPTIMISATIONS v6 :
#   ✦ [OPT-1] Décodage uniquement des tokens GÉNÉRÉS (slice input_len:)
#             → évite de décoder le prompt entier à chaque page
#   ✦ [OPT-2] min_pixels / max_pixels passés au processor Qwen2-VL
#             → budget visuel fixe, pas de patches supplémentaires
#   ✦ [OPT-3] Image.resize avec LANCZOS → filtre rapide et net
#   ✦ [OPT-4] Sauvegarde incrémentale en mode "append-line" (JSONL)
#             + reconstruction JSON complète uniquement à la fin
#             → évite de réécrire tout le fichier après chaque page
#   ✦ [OPT-5] torch.compile(model) si PyTorch >= 2.0 et CUDA présent
#             → accélération ~15-30 % sur T4 après la 1ère page
#
# INTERFACE DE SORTIE :
#   ocr_pages  : dict[int, str]   (variable en mémoire)
#   OCR_CACHE  : "/content/loc_ocr_pages.json"  (sur disque, JSON)
#   OCR_CACHE_JSONL : "/content/loc_ocr_pages.jsonl" (log incrémental)
# ═══════════════════════════════════════════════════════════════════

import os, json, glob, subprocess, time, re
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

OCR_CACHE      = "/content/loc_ocr_pages.json"
OCR_CACHE_JSONL = "/content/loc_ocr_pages.jsonl"  # [OPT-4] log incrémental
IMG_DIR        = "/content/loc_pages"
PDF_PATH       = "/content/drive/MyDrive/3D_SMART/lois/les_lois_des_obligations_et_contrats.pdf"

# [OPT-2] Budget de patches visuels pour Qwen2-VL
# 256×256 px minimum, 1024×1024 px maximum → ~256 patches max
MIN_PIXELS = 256 * 28 * 28   #  ~200 k pixels
MAX_PIXELS = 1280 * 28 * 28  # ~1 M pixels  (valeur recommandée Qwen2-VL)

os.makedirs(IMG_DIR, exist_ok=True)


# ═══════════════════════════════════════════════════════════════════
# MODÈLE
# ═══════════════════════════════════════════════════════════════════

model_name = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"
device     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

# [OPT-2] Passer min/max_pixels au processor pour contraindre le budget visuel
processor = AutoProcessor.from_pretrained(
    model_name,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

# [OPT-5] torch.compile — accélère de ~15-30 % sur GPU après warm-up
if device == "cuda" and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("  torch.compile activé ✓")
    except Exception as e:
        print(f"  torch.compile ignoré : {e}")


# ═══════════════════════════════════════════════════════════════════
# SECTION 1 — RASTERISATION PDF
# ═══════════════════════════════════════════════════════════════════

def rasterize_pdf(pdf_path: str, out_dir: str, dpi: int = 100) -> list[str]:
    """Convertit le PDF en images JPEG, une par page."""
    existing = sorted(glob.glob(f"{out_dir}/page-*.jpg"))
    if existing:
        print(f"  {len(existing)} pages déjà présentes — réutilisation du cache image")
        return existing
    print(f"  Rasterisation PDF ({dpi} DPI)…")
    subprocess.run(
        ["pdftoppm", "-jpeg", "-r", str(dpi), "-jpegopt", "quality=80",
         pdf_path, f"{out_dir}/page"],
        check=True
    )
    pages = sorted(glob.glob(f"{out_dir}/page-*.jpg"))
    print(f"  {len(pages)} pages générées")
    return pages


# ═══════════════════════════════════════════════════════════════════
# SECTION 2 — OCR PAR PAGE (avec cache)
# ═══════════════════════════════════════════════════════════════════

def _ocr_single_page(image_path: str) -> str:
    """
    OCR d'une seule page — retourne le texte brut.

    [OPT-1] On ne décode que les tokens générés (output[:, input_len:])
            et non le prompt entier, ce qui divise le temps de décodage.
    [OPT-3] PIL resize avec LANCZOS (meilleur rapport qualité/vitesse).
    """
    # [OPT-3] Filtre LANCZOS explicite
    image = Image.open(image_path).convert("RGB").resize(
        (896, 896), Image.LANCZOS
    )
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": "Extract all text from this page."}
        ]
    }]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image], return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]  # [OPT-1] longueur du prompt

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            temperature=None,   # désactive le warning transformers >= 4.40
            top_p=None,
        )

    # [OPT-1] Décoder uniquement les tokens nouveaux
    generated = output[:, input_len:]
    text = processor.batch_decode(generated, skip_special_tokens=True)[0]

    del inputs, output, generated
    torch.cuda.empty_cache()
    return text.strip()


def _load_jsonl_cache(jsonl_path: str) -> dict:
    """
    [OPT-4] Recharge le cache depuis le fichier JSONL incrémental.
    Chaque ligne est {"page": N, "text": "..."}.
    En cas de ligne corrompue (crash mid-write), elle est ignorée.
    """
    results = {}
    if not os.path.exists(jsonl_path):
        return results
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
                results[str(entry["page"])] = entry["text"]
            except (json.JSONDecodeError, KeyError):
                pass  # ligne corrompue → ignorée
    return results


def run_ocr_with_cache(
    pdf_path:   str,
    img_dir:    str,
    cache_path: str,
    dpi:        int  = 100,
    force:      bool = False,
) -> dict[int, str]:
    """
    Rasterise le PDF, OCRise chaque page et retourne un dict
    {page_num: texte_brut}.

    [OPT-4] Sauvegarde incrémentale en JSONL (une ligne par page) :
    - Chaque page traitée est appendée au fichier JSONL → écriture O(1)
    - Le JSON final est reconstruit une seule fois à la fin
    - En cas d'interruption, la reprise relit le JSONL existant
    """
    jsonl_path = cache_path.replace(".json", ".jsonl")

    if force:
        for p in [cache_path, jsonl_path]:
            if os.path.exists(p):
                os.remove(p)
                print(f"  Cache supprimé : {p}")
        results = {}
    elif os.path.exists(cache_path):
        # Cache JSON complet disponible → chargement rapide
        with open(cache_path, encoding="utf-8") as f:
            results = json.load(f)
        print(f"  Cache JSON chargé : {len(results)} pages")
    else:
        # [OPT-4] Pas de JSON complet → essayer de reprendre depuis JSONL
        results = _load_jsonl_cache(jsonl_path)
        if results:
            print(f"  Cache JSONL repris : {len(results)} pages déjà traitées")
        else:
            results = {}

    pages = rasterize_pdf(pdf_path, img_dir, dpi=dpi)
    todo  = [p for p in pages
             if str(int(re.search(r'page-(\d+)\.jpg', p).group(1))) not in results]

    if not todo:
        print("  Toutes les pages sont dans le cache ✓")
        return {int(k): v for k, v in results.items()}

    print(f"  Pages restantes à OCRiser : {len(todo)}")
    start    = time.time()
    n_done   = 0
    eta_secs = None

    # [OPT-4] Ouvrir le JSONL en mode append pour écriture incrémentale
    with open(jsonl_path, "a", encoding="utf-8") as jsonl_file:
        for f in todo:
            page_num = int(re.search(r'page-(\d+)\.jpg', f).group(1))
            eta_str  = f"  ETA ~{eta_secs/60:.1f} min" if eta_secs else ""
            print(f"  Page {page_num:3d}…{eta_str}", end=" ", flush=True)

            t0 = time.time()
            try:
                text = _ocr_single_page(f)
                results[str(page_num)] = text
                status = f"{len(text):5d} chars"
            except Exception as e:
                text = f"[ERROR {page_num}: {e}]"
                results[str(page_num)] = text
                status = f"ERROR: {e}"
                torch.cuda.empty_cache()

            page_time = time.time() - t0
            print(f"{status}  [{page_time:.1f}s]")

            # [OPT-4] Append d'une seule ligne JSON → O(1) par page
            jsonl_file.write(
                json.dumps({"page": page_num, "text": text}, ensure_ascii=False) + "\n"
            )
            jsonl_file.flush()

            n_done += 1
            elapsed  = time.time() - start
            remaining = len(todo) - n_done
            eta_secs  = (elapsed / n_done) * remaining if n_done > 0 else None

    # [OPT-4] Reconstruction du JSON complet une seule fois à la fin
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n  OCR terminé en {(time.time()-start)/60:.2f} min")
    print(f"  Cache JSON reconstruit → {cache_path}")

    return {int(k): v for k, v in results.items()}


# ═══════════════════════════════════════════════════════════════════
# POINT D'ENTRÉE — Cell 2
# ═══════════════════════════════════════════════════════════════════

print("━" * 60)
print("CELL 2 — OCR brut des pages")
print("━" * 60)

assert os.path.exists(PDF_PATH), f"PDF introuvable : {PDF_PATH}"

ocr_pages = run_ocr_with_cache(
    pdf_path   = PDF_PATH,
    img_dir    = IMG_DIR,
    cache_path = OCR_CACHE,
    dpi        = 100,
    force      = False,   # True pour forcer une nouvelle passe complète
)

print(f"\n  Pages disponibles : {len(ocr_pages)}")
print(f"  Cache OCR brut   → {OCR_CACHE}")
print("\n  ✓ Cell 2 terminée — exécutez Cell 3 pour l'analyse structurelle.")

Device : cuda


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

  torch.compile activé ✓
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELL 2 — OCR brut des pages
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Cache JSON chargé : 172 pages
  268 pages déjà présentes — réutilisation du cache image
  Pages restantes à OCRiser : 96
  Page 173…  1776 chars  [42.7s]
  Page 174…  ETA ~67.6 min  1961 chars  [47.2s]
  Page 175…  ETA ~70.5 min  1987 chars  [46.5s]
  Page 176…  ETA ~70.5 min  1662 chars  [46.6s]
  Page 177…  ETA ~70.1 min  1935 chars  [47.5s]
  Page 178…  ETA ~69.9 min  1946 chars  [46.8s]
  Page 179…  ETA ~69.3 min  2069 chars  [47.7s]
  Page 180…  ETA ~68.9 min  2015 chars  [46.2s]
  Page 181…  ETA ~68.0 min  1624 chars  [37.8s]
  Page 182…  ETA ~65.9 min  2022 chars  [46.2s]
  Page 183…  ETA ~65.2 min  2004 chars  [46.4s]
  Page 184…  ETA ~64.6 min  1988 chars  [46.5s]
  Page 185…  ETA ~63.9 min  1783 chars  [42.1s]
  Page 186…  ETA ~62.8 min  2018 chars  [46.8s]
  Page 187…  ETA ~62.2 min  1936 chars  [46.1

In [30]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 (v5) + CELL 4 (v5) — Pipeline complet
#
# ROOT CAUSE de الفصل 3 → 48 pages :
#   analyse_page() marquait systématiquement le dernier article d'une page
#   comme "incomplet" si sa dernière ligne ne finissait pas par ponctuation.
#   الفصل 3 a un corps court (2 lignes), sa dernière ligne se termine par
#   un chiffre de note de bas de page (ex: "…بغير ذلك 16.") — la regex
#   _SENTENCE_END détectait le "." MAIS _TRAILING_DIGIT prenait la priorité
#   et forçait complete=False → fusion sur 48 pages.
#
# CORRECTIFS v5 :
#   F1 — _article_seems_complete() : le "." terminal l'emporte sur le chiffre
#        de note de bas de page (pattern "texte 16." → complet)
#   F2 — analyse_page() : article avec corps ≤ 2 lignes → complet=True
#        (un article de 2 lignes qui déborde sur 48 pages est impossible)
#   F3 — rebuild_cross_page_articles() : garde-fou densité الفصل N sur
#        la page suivante (> MAX_FASIL_ON_NEXT_PAGE → stop fusion)
#   F4 — _purge_body_lines() : élimine headers institutionnels dans le corps
#
# CELL 4 v5 :
#   • Consomme directement structured['articles'] + chunks (pas l'OCR brut)
#   • Supprime le cache périmé au démarrage
#   • _clean_body_v5() filtre les parasites résiduels
# ══════════════════════════════════════════════════════════════════════════════

import os, re, json, unicodedata
from dataclasses import dataclass, asdict
from typing import Optional

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────

OCR_CACHE      = "/content/loc_ocr_pages.json"
STRUCT_OUT     = "/content/loc_structured.json"
RAG_OUT        = "/content/loc_rag_chunks.json"
ARTICLES_CACHE = "/content/loc_articles.json"

MAX_CONTINUATION_LINES    = 20
MAX_CONTINUATION_PAGES    = 2      # réduit de 3 → 2
MAX_BODY_LINES_SHORT_ART  = 2      # F2 : ≤ N lignes → complet immédiat
MAX_FASIL_ON_NEXT_PAGE    = 4      # F3 : densité الفصل sur page suivante
MIN_ARABIC_CHARS          = 15
LOC_MAX_NUM               = 1260


# ════════════════════════════════════════════════════════════════════
# ── NORMALISATION HTML OCR ─────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════

def _normalize_html(raw: str) -> str:
    # P1
    raw = re.sub(
        r'(<h[1-6][^>]*>)\s*<(?:u|b|i)>\s*(الفصل)\s*</(?:u|b|i)>\s+'
        r'(\d{1,4}(?:[.\-]\d+)?(?:\s+مكرر)?)\s*(</h[1-6]>)',
        r'<h2>\2 \3</h2>', raw, flags=re.UNICODE)
    # P2
    raw = re.sub(
        r'(<h[1-6][^>]*>)\s*(\d{1,4})\s+الفصل\s*(</h[1-6]>)',
        r'<h2>الفصل \2</h2>', raw, flags=re.UNICODE)
    # P3
    raw = re.sub(
        r'(<h[1-6][^>]*>)\s*(\d{1,4})\s*<(?:i|b|u)>\s*الفصل\s*</(?:i|b|u)>\s*(</h[1-6]>)',
        r'<h2>الفصل \2</h2>', raw, flags=re.UNICODE)
    # P4a
    raw = re.sub(
        r'<p[^>]*>\s*(?:\d+\s+)?<b>الفصل</b>\s+(\d{1,4}(?:[.\-]\d+)?)\s*(.*?)</p>',
        lambda m: (f'<h2>الفصل {m.group(1)}</h2>'
                   + (f'<p>{m.group(2).strip()}</p>' if m.group(2).strip() else '')),
        raw, flags=re.DOTALL | re.UNICODE)
    # P4b
    raw = re.sub(
        r'<p[^>]*>\s*\d+\s+<b>الفصل</b>\s+(\d{1,4}(?:[.\-]\d+)?)\s+(.*?)</p>',
        lambda m: (f'<h2>الفصل {m.group(1)}</h2>'
                   + (f'<p>{m.group(2).strip()}</p>' if m.group(2).strip() else '')),
        raw, flags=re.DOTALL | re.UNICODE)
    # P5
    raw = re.sub(
        r'<p[^>]*>\s*(الفصل\s+\d{1,4}(?:[.\-]\d+)?)\s*</p>',
        r'<h2>\1</h2>', raw, flags=re.UNICODE)
    # P6a
    raw = re.sub(
        r'<h[1-6][^>]*>\s*(\d{1,4})\s*</h[1-6]>\s*<br>\s*<h2>(الفصل\s+\1)</h2>',
        r'<h2>\2</h2>', raw, flags=re.UNICODE)
    # P6b
    raw = re.sub(
        r'<h[1-6][^>]*>\s*(\d{1,4})\s*</h[1-6]>\s*(?:<br>)?\s*<p[^>]*>\s*الفصل\s*</p>',
        r'<h2>الفصل \1</h2>', raw, flags=re.UNICODE)
    # P6c
    raw = re.sub(
        r'<h[1-6][^>]*>\s*(\d{1,4})\s*</h[1-6]>',
        r'<h2>الفصل \1</h2>', raw, flags=re.UNICODE)
    # orphelins
    raw = re.sub(r'<p[^>]*>\s*الفصل\s*</p>', '', raw, flags=re.UNICODE)
    # inline dans h
    raw = re.sub(
        r'<(h[1-6])[^>]*>\s*(الفصل\s+\d{1,4}(?:[.\-]\d+)?)\s+([^<]{15,})</\1>',
        r'<h2>\2</h2><p>\3</p>', raw, flags=re.UNICODE)
    return raw


def _html_to_text(raw: str) -> str:
    m = re.search(r'\bassistant\s*\n', raw, re.IGNORECASE)
    raw = raw[m.end():] if m else raw
    raw = _normalize_html(raw)
    raw = re.sub(r'<br\s*/?>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'</(h[1-6]|p|div|li|ul|ol|tr|td|th)>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<(h[1-6]|p|div|li|ul|ol|tr|td|th)[^>]*>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<[^>]+>', ' ', raw)
    raw = re.sub(r'[ \t]+', ' ', raw)
    raw = re.sub(r'\n{3,}', '\n\n', raw)
    return raw.strip()


# ════════════════════════════════════════════════════════════════════
# ── ANALYSE STRUCTURELLE ───────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════

def detect_arabic_ratio(text: str) -> float:
    cleaned = re.sub(r'\s', '', text)
    if not cleaned:
        return 0.0
    arabic = sum(1 for c in cleaned if '\u0600' <= c <= '\u06FF')
    return round(arabic / len(cleaned), 3)

_HEADER_FOOTER_PAT = re.compile(
    r'(?:المملكة\s+المغربية|وزارة\s+(?:العدل|العدار|العدد)'
    r'|مديرية\s+التشريع|قانون\s+الالتزامات\s+والعقود|صيغة\s+محينة)',
    re.UNICODE)
_PAGE_NUMBER_PAT = re.compile(r'^\s*-?\s*\d{1,3}\s*-?\s*$')

def extract_header_footer(lines: list) -> dict:
    non_empty = [l.strip() for l in lines if l.strip()]
    if not non_empty:
        return {"header": [], "footer": [], "page_number": None}
    headers  = [l for l in non_empty[:3]  if _HEADER_FOOTER_PAT.search(l)]
    footers  = [l for l in non_empty[-3:] if _HEADER_FOOTER_PAT.search(l)]
    page_num = None
    for l in non_empty[:4] + non_empty[-4:]:
        if _PAGE_NUMBER_PAT.match(l):
            try:
                page_num = int(re.search(r'\d+', l).group())
            except Exception:
                pass
            break
    return {"header": headers, "footer": footers, "page_number": page_num}

_NOTE_LINE_START = re.compile(r'^\s*\d{1,2}\s*[-–)]\s+', re.UNICODE)
_NOTE_KEYWORD    = re.compile(
    r'(?:الجريدة\s+الرسمية|ظهير\s+(?:شريف|رقم)|صادر\s+في\s+\d+'
    r'|المؤرخ\s+(?:في\s+)?\d|القانون\s+رقم\s+\d+[\.\-]\d+'
    r'|المرسوم\s+رقم|القرار\s+رقم)', re.UNICODE)

def extract_footnotes(lines: list) -> tuple:
    in_footnote = False
    content, notes = [], []
    for line in lines:
        stripped = line.strip()
        if not in_footnote:
            if _NOTE_LINE_START.match(stripped) and _NOTE_KEYWORD.search(stripped):
                in_footnote = True
        (notes if in_footnote else content).append(line)
    return content, notes

_STRUCT_LEVELS = {'الكتاب': 1, 'القسم': 2, 'الباب': 3, 'الفرع': 4}
_STRUCT_PAT    = re.compile(r'^(?P<level>الكتاب|القسم|الباب|الفرع)\s+(?P<rest>.+)', re.UNICODE)

def classify_structural_line(line: str) -> Optional[dict]:
    m = _STRUCT_PAT.match(line.strip())
    if not m:
        return None
    kw = m.group('level')
    return {"level_num": _STRUCT_LEVELS.get(kw, 0), "keyword": kw, "text": line.strip()}

_ARABIC_INDIC_MAP = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
_ARTICLE_PAT      = re.compile(
    r'^الفصل[\s\u00a0\t]+'
    r'(?P<id>\d{1,4}(?:[.\-]\d{1,4})?(?:\s+مكرر)?)'
    r'(?:[\s\u00a0]+(?P<note>\d{1,3}))?'
    r'(?:[\s\u00a0]+(?P<inline>.+))?$',
    re.UNICODE | re.DOTALL)

def _normalize_article_id(raw_id: str) -> str:
    raw_id = raw_id.strip().replace('.', '-')
    if '-' not in raw_id:
        return raw_id
    parts = raw_id.split('-', 1)
    try:
        a, b = int(parts[0]), int(parts[1])
        return f"{max(a,b)}-{min(a,b)}"
    except ValueError:
        return raw_id

def parse_article_line(line: str) -> Optional[dict]:
    line = re.sub(r'^الفصيل\s*', 'الفصل ', line.strip())
    line = line.replace('\u00a0', ' ').replace('\u2011', '-')
    line = line.translate(_ARABIC_INDIC_MAP)
    line = re.sub(r'^(الفصل)[\s\t]{2,}', r'\1 ', line)
    m = _ARTICLE_PAT.match(line.strip())
    if not m:
        return None
    raw_id     = m.group('id').strip()
    note_ref   = m.group('note')
    inline_raw = (m.group('inline') or '').strip()
    norm_id    = _normalize_article_id(raw_id)
    if note_ref and inline_raw.startswith(note_ref):
        inline_raw = inline_raw[len(note_ref):].strip()
    arabic_count   = sum(1 for c in inline_raw if '\u0600' <= c <= '\u06FF')
    inline_content = inline_raw if arabic_count > 3 else ''
    return {"raw_id": raw_id, "normalized_id": norm_id,
            "note_ref": note_ref, "inline_content": inline_content}

_INTERNAL_REF_PAT = re.compile(
    r'(?:(?:الفصل|الفصول)\s+(?:\d{1,4}(?:[.\-]\d{1,4})?)'
    r'(?:\s+(?:و|إلى)\s+(?:\d{1,4}(?:[.\-]\d{1,4})?))*)', re.UNICODE)

def extract_internal_refs(text: str) -> list:
    return list(dict.fromkeys(_INTERNAL_REF_PAT.findall(text)))

def _split_structural_and_articles(line: str) -> list:
    section_kw = r'(?:الكتاب|القسم|الباب|الفرع)'
    parts = re.split(r'(?<!\S)(' + section_kw + r')', line)
    pieces = []
    i = 0
    while i < len(parts):
        if i == 0:
            piece = parts[i].strip()
        else:
            piece = (parts[i] + (parts[i+1] if i+1 < len(parts) else '')).strip()
            i += 1
        i += 1
        if not piece:
            continue
        sub = re.split(
            r'(?<!\S)((?:الفصل|الفصيل)[\s\u00a0\t]+\d[\d.\-]*(?:\s+مكرر)?(?:[\s\u00a0]+\d{1,3})?)',
            piece)
        for sp in sub:
            if sp.strip():
                pieces.append(sp.strip())
    return pieces or [line]

def build_page_lines(raw_text: str) -> list:
    text = _html_to_text(raw_text)
    text = text.translate(_ARABIC_INDIC_MAP)
    lines = []
    for raw_line in text.split('\n'):
        for part in _split_structural_and_articles(raw_line.strip()):
            part = re.sub(r'^الفصيل\s*', 'الفصل ', part).strip()
            part = re.sub(r'^(الفصل)[\s\t]{2,}(\d)', r'\1 \2', part)
            if part:
                lines.append(part)
    return lines


# ── F1 : _article_seems_complete corrigé ──────────────────────────
# Avant : "…بغير ذلك 16." → _TRAILING_DIGIT prenait la priorité → False
# Après : si la ligne finit par "chiffre + ponctuation", c'est complet
_SENTENCE_END        = re.compile(r'[.؟!،؛:۔]\s*$', re.UNICODE)
_TRAILING_DIGIT_ONLY = re.compile(r'\d+\s*$', re.UNICODE)
# Note de bas de page : chiffre PRÉCÉDÉ de ponctuation (ex: "ذلك 16.")
_NOTE_THEN_PUNCT     = re.compile(r'[.؟!،؛]\s*\d{1,3}\s*[.؟!،؛]?\s*$', re.UNICODE)
_ENDS_DIGIT_NO_PUNCT = re.compile(r'(?<![.؟!،؛])\s*\d+\s*$', re.UNICODE)

def _article_seems_complete(lines: list) -> bool:
    last = ''
    for line in reversed(lines):
        stripped = line.strip()
        if stripped:
            last = stripped
            break
    if not last:
        return True

    # Cas F1 : "texte. 16" ou "texte 16." → note de bas de page → complet
    if _NOTE_THEN_PUNCT.search(last):
        return True
    # Chiffre final SANS ponctuation → vraiment incomplet
    if _ENDS_DIGIT_NO_PUNCT.search(last):
        return False

    arabic_chars = sum(1 for c in last if '\u0600' <= c <= '\u06FF')
    if arabic_chars > 80:
        return True
    if arabic_chars > 0 and len(last) < 25 and not _SENTENCE_END.search(last):
        return False
    return bool(_SENTENCE_END.search(last))


@dataclass
class PageAnalysis:
    page_num:            int
    arabic_ratio:        float
    is_arabic:           bool
    header:              list
    footer:              list
    detected_page_num:   Optional[int]
    structural_elements: list
    articles:            list
    footnotes:           list
    internal_refs:       list
    raw_text:            str
    content_lines:       list
    is_error:            bool = False
    error_msg:           str  = ""


def analyse_page(page_num: int, raw_ocr: str) -> PageAnalysis:
    if raw_ocr.startswith("[ERROR"):
        return PageAnalysis(
            page_num=page_num, arabic_ratio=0.0, is_arabic=False,
            header=[], footer=[], detected_page_num=None,
            structural_elements=[], articles=[], footnotes=[],
            internal_refs=[], raw_text=raw_ocr, content_lines=[],
            is_error=True, error_msg=raw_ocr)

    lines        = build_page_lines(raw_ocr)
    full_text    = '\n'.join(lines)
    arabic_ratio = detect_arabic_ratio(full_text)
    is_arabic    = arabic_ratio >= 0.25
    hf           = extract_header_footer(lines)
    content_lines, footnote_lines = extract_footnotes(lines)

    struct_elements = []
    articles        = []
    current_article = None

    for line in content_lines:
        struct = classify_structural_line(line)
        if struct:
            struct_elements.append(struct)
            if current_article:
                current_article["complete"] = _article_seems_complete(current_article["body"])
                articles.append(current_article)
                current_article = None
            continue
        art = parse_article_line(line)
        if art:
            if current_article:
                current_article["complete"] = _article_seems_complete(current_article["body"])
                articles.append(current_article)
            header_line = line.strip()
            if art["note_ref"]:
                header_line = re.sub(
                    r'\s+' + re.escape(art["note_ref"]) + r'\s*$', '', header_line).strip()
            current_article = {
                "normalized_id":  art["normalized_id"],
                "raw_id":         art["raw_id"],
                "note_ref":       art["note_ref"],
                "header_line":    header_line,
                "body":           [],
                "complete":       False,
                "starts_on_page": page_num,
            }
            if art["inline_content"]:
                current_article["body"].append(art["inline_content"])
            continue
        if current_article and line.strip():
            current_article["body"].append(line)

    if current_article:
        body = current_article["body"]
        # ── F2 : article court (≤ MAX_BODY_LINES_SHORT_ART lignes non-vides) ──
        non_empty = [l for l in body if l.strip()]
        if len(non_empty) <= MAX_BODY_LINES_SHORT_ART:
            current_article["complete"] = True
        else:
            current_article["complete"] = _article_seems_complete(body)
        articles.append(current_article)

    return PageAnalysis(
        page_num=page_num, arabic_ratio=arabic_ratio, is_arabic=is_arabic,
        header=hf["header"], footer=hf["footer"],
        detected_page_num=hf["page_number"],
        structural_elements=struct_elements, articles=articles,
        footnotes=footnote_lines, internal_refs=extract_internal_refs(full_text),
        raw_text=full_text, content_lines=content_lines)


def _count_fasil_on_page(page: PageAnalysis) -> int:
    return len(re.findall(r'الفصل\s+\d', page.raw_text))

def _get_continuation_lines(next_page: PageAnalysis,
                             max_lines: int = MAX_CONTINUATION_LINES) -> list:
    continuation = []
    for line in next_page.content_lines:
        if classify_structural_line(line):
            break
        if parse_article_line(line):
            break
        if _HEADER_FOOTER_PAT.search(line):
            continue
        if _PAGE_NUMBER_PAT.match(line.strip()):
            continue
        if line.strip():
            continuation.append(line)
            if len(continuation) >= max_lines:
                break
    return continuation

def _next_page_starts_with_new_article(next_page: PageAnalysis) -> bool:
    real_lines_seen = 0
    for line in next_page.content_lines:
        stripped = line.strip()
        if not stripped:
            continue
        if _HEADER_FOOTER_PAT.search(stripped):
            continue
        if _PAGE_NUMBER_PAT.match(stripped):
            continue
        if classify_structural_line(stripped) or parse_article_line(stripped):
            return True
        real_lines_seen += 1
        if real_lines_seen >= 5:
            return False
    return False

def rebuild_cross_page_articles(pages: list) -> dict:
    sorted_pages = sorted(pages, key=lambda p: p.page_num)
    page_map     = {p.page_num: p for p in sorted_pages}
    merged_arts  = {}

    for page in sorted_pages:
        for art in page.articles:
            aid = art["normalized_id"]
            if aid not in merged_arts:
                merged_arts[aid] = dict(art)
            else:
                existing = merged_arts[aid]
                if not existing["complete"]:
                    existing["body"].extend(art["body"])
                    existing["complete"]     = art["complete"]
                    existing["ends_on_page"] = page.page_num

    for page in sorted_pages:
        if not page.articles:
            continue
        last_art = page.articles[-1]
        aid      = last_art["normalized_id"]
        merged   = merged_arts.get(aid)
        if merged is None or merged.get("complete", True):
            continue

        continuation_pages_used = 0
        current_page_num        = page.page_num

        while (not merged.get("complete", True)
               and continuation_pages_used < MAX_CONTINUATION_PAGES):
            next_page = page_map.get(current_page_num + 1)
            if next_page is None or next_page.is_error:
                break
            # ── F3 : densité الفصل N sur la page suivante ──────────
            if _count_fasil_on_page(next_page) > MAX_FASIL_ON_NEXT_PAGE:
                merged["complete"]     = True
                merged["ends_on_page"] = current_page_num
                break
            if _next_page_starts_with_new_article(next_page):
                merged["complete"]     = True
                merged["ends_on_page"] = current_page_num
                break
            continuation = _get_continuation_lines(next_page)
            if not continuation:
                merged["complete"]     = True
                merged["ends_on_page"] = current_page_num
                break
            merged["body"].extend(continuation)
            merged["ends_on_page"] = next_page.page_num
            merged["complete"]     = _article_seems_complete(merged["body"])
            continuation_pages_used += 1
            current_page_num        = next_page.page_num

        if continuation_pages_used >= MAX_CONTINUATION_PAGES and not merged.get("complete"):
            merged["complete"]     = True
            merged["ends_on_page"] = current_page_num

    for art in merged_arts.values():
        if "ends_on_page" not in art:
            art["ends_on_page"] = art["starts_on_page"]

    return merged_arts

def build_hierarchy_context(pages: list) -> dict:
    context_stack = []
    art_contexts  = {}
    for page in sorted(pages, key=lambda p: p.page_num):
        for struct in page.structural_elements:
            level = struct["level_num"]
            context_stack = [c for c in context_stack if c["level_num"] < level]
            context_stack.append(struct)
        for art in page.articles:
            art_contexts[art["normalized_id"]] = " > ".join(
                c["text"] for c in context_stack)
    return art_contexts

def _sort_key(k: str) -> tuple:
    parts = k.split('-')
    try:
        if len(parts) == 2:
            a, b = int(parts[0]), int(parts[1])
            return (max(a, b), min(a, b))
        return (int(parts[0]), 0)
    except ValueError:
        return (9999, 0)

# ── F4 : purge des parasites dans les corps exportés ──────────────
_INSTITUTIONAL_BODY = re.compile(
    r'(?:المملكة\s+المغربية|وزارة\s+(?:العدل|العدار|العدد)'
    r'|مديرية\s+التشريع|قانون\s+الالتزامات\s+والعقود|صيغة\s+محينة)',
    re.UNICODE)

def _purge_body_lines(lines: list) -> list:
    return [
        l for l in lines
        if not _INSTITUTIONAL_BODY.search(l)
        and not _PAGE_NUMBER_PAT.match(l.strip())
    ]

def export_structured(pages: list, merged_articles: dict,
                      art_contexts: dict, output_path: str) -> dict:
    arabic_pages = sum(1 for p in pages if p.is_arabic)
    all_refs     = []
    for p in pages:
        all_refs.extend(p.internal_refs)
    unique_refs = list(dict.fromkeys(all_refs))

    metadata = {
        "total_pages":       len(pages),
        "arabic_pages":      arabic_pages,
        "total_articles":    len(merged_articles),
        "all_internal_refs": unique_refs,
    }
    pages_export = []
    for p in sorted(pages, key=lambda x: x.page_num):
        pages_export.append({
            "page_num":            p.page_num,
            "arabic_ratio":        p.arabic_ratio,
            "is_arabic":           p.is_arabic,
            "detected_page_num":   p.detected_page_num,
            "header":              p.header,
            "footer":              p.footer,
            "structural_elements": p.structural_elements,
            "articles_on_page":    [a["normalized_id"] for a in p.articles],
            "footnotes_count":     len(p.footnotes),
            "internal_refs":       p.internal_refs,
            "is_error":            p.is_error,
        })

    articles_export = {}
    for aid, art in sorted(merged_articles.items(), key=lambda x: _sort_key(x[0])):
        body_lines = _purge_body_lines([l for l in art.get("body", []) if l.strip()])
        contenu    = '\n'.join(body_lines)
        if art.get("note_ref"):
            contenu = re.sub(
                r'^\s*' + re.escape(str(art["note_ref"])) + r'\s*\n?', '', contenu).strip()
        articles_export[aid] = {
            "normalized_id":  aid,
            "header_line":    art.get("header_line", f"الفصل {aid}"),
            "contexte":       art_contexts.get(aid, ""),
            "contenu":        contenu,
            "starts_on_page": art.get("starts_on_page"),
            "ends_on_page":   art.get("ends_on_page"),
            "complete":       art.get("complete", True),
            "note_ref":       art.get("note_ref"),
            "internal_refs":  extract_internal_refs(contenu),
        }

    out = {"metadata": metadata, "pages": pages_export, "articles": articles_export}
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    return out


# ════════════════════════════════════════════════════════════════════
# ── NETTOYAGE RAG ─────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════

_BIDI_AND_FORMAT   = re.compile(r'[\u200b-\u200f\u202a-\u202e\u2060-\u2064\u061c\ufeff\u00ad]')
_SPECIAL_SPACES    = re.compile(r'[\u00a0\u202f\u2007-\u200a\u3000]')
_TATWEEL           = re.compile(r'\u0640')
_HARAKAT           = re.compile(
    r'[\u064b-\u065f\u0610-\u061a\u06d6-\u06dc\u06df-\u06e4\u06e7\u06e8\u06ea-\u06ed\u0670]')
_PERSIAN_TO_ARABIC = str.maketrans({
    '\u06a9': '\u0643', '\u06cc': '\u064a', '\u06c1': '\u0647',
    '\u06be': '\u0647', '\u06d5': '\u0647', '\u0649': '\u064a'})
_ARABIC_INDIC_DIGITS = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')

def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize('NFC', text)
    text = _BIDI_AND_FORMAT.sub('', text)
    text = _SPECIAL_SPACES.sub(' ', text)
    text = _TATWEEL.sub('', text)
    text = _HARAKAT.sub('', text)
    text = text.translate(_PERSIAN_TO_ARABIC)
    text = text.translate(_ARABIC_INDIC_DIGITS)
    return text

_FASIL_FIX           = re.compile(r'\bالفصيل\b', re.UNICODE)
_ARABIC_HYPHEN_BREAK = re.compile(r'([\u0600-\u06ff])-\s*\n\s*([\u0600-\u06ff])', re.UNICODE)
_FASL_DOUBLE_SPACE   = re.compile(r'^(الفصل)\s{2,}(\d)', re.UNICODE | re.MULTILINE)
_OCR_FIXES_LITERAL   = {
    'بوجهعام': 'بوجه عام', 'محلاللتزام': 'محل الالتزام',
    'بغيرحق': 'بغير حق', 'الناشئةعن': 'الناشئة عن',
    'أشباهالجرائم': 'أشباه الجرائم', 'أشباهالعقود': 'أشباه العقود',
    'مالم': 'ما لم', 'بشرطأن': 'بشرط أن', 'علىأن': 'على أن',
    'إلاإذا': 'إلا إذا', 'الإستحقاق': 'الاستحقاق',
    'الإستيفاء': 'الاستيفاء', 'الإستعمال': 'الاستعمال',
    'الناشنة': 'الناشئة', 'القنصوص': 'المنصوص',
    'اليريدة': 'الجريدة', 'الإراده': 'الإرادة',
    'الأهليه': 'الأهلية', 'المدنيه': 'المدنية',
    'المسؤوليه': 'المسؤولية', 'الملكيه': 'الملكية',
}
_OCR_FIXES_COMPILED = [
    (re.compile(r'\b' + re.escape(k) + r'\b', re.UNICODE), v)
    for k, v in _OCR_FIXES_LITERAL.items()]

def apply_ocr_fixes(text: str) -> str:
    text = _FASIL_FIX.sub('الفصل', text)
    text = _ARABIC_HYPHEN_BREAK.sub(r'\1\2', text)
    text = _FASL_DOUBLE_SPACE.sub(r'\1 \2', text)
    for pattern, replacement in _OCR_FIXES_COMPILED:
        text = pattern.sub(replacement, text)
    return text

_PAGE_NUMBER_RAG = re.compile(r'^\s*-?\s*\d{1,3}\s*-?\s*$')
_DOTTED_LINE     = re.compile(r'^[\s.\-_=~*]{5,}$')
_FOOTNOTE_BLOCK  = re.compile(
    r'(?:الجريدة\s+الرسمية|ظهير\s+(?:شريف|رقم)\s|صادر\s+في\s+\d'
    r'|المؤرخ\s+(?:في\s+)?\d|القانون\s+رقم\s+\d+[\.\-]\d+'
    r'|المرسوم\s+رقم\s|القرار\s+رقم\s)', re.UNICODE)
_PURE_NON_ARABIC = re.compile(r'^[\d\s\.\,\:\;\!\?\-\(\)\[\]\/\\\"\']+$')

def is_junk_line(line: str) -> bool:
    s = line.strip()
    if not s:
        return True
    if _PAGE_NUMBER_RAG.match(s) or _DOTTED_LINE.match(s):
        return True
    if _INSTITUTIONAL_BODY.search(s) or _FOOTNOTE_BLOCK.search(s):
        return True
    if _PURE_NON_ARABIC.match(s):
        return True
    arabic_count = sum(1 for c in s if '\u0600' <= c <= '\u06ff')
    if len(s) > 2 and arabic_count == 0:
        return True
    return False

_MULTI_SPACE   = re.compile(r'[ \t]+')
_MULTI_NEWLINE = re.compile(r'\n{3,}')
_TRAILING_WS   = re.compile(r'[ \t]+$', re.MULTILINE)
_LEADING_WS    = re.compile(r'^[ \t]+', re.MULTILINE)
_NOTE_INLINE   = re.compile(
    r'(?<=[^\s\d\u0660-\u0669])(\d{1,2})(?=[\s\.\،\؛\:\!\؟\)]|$)', re.UNICODE)
_GUILLEMETS    = re.compile(r'[«»""‟„\u201c\u201d\u201e\u201f]')
_EM_DASH       = re.compile(r'[—–\u2013\u2014\u2015]')

def clean_text(raw: str, is_body: bool = True) -> str:
    if not raw:
        return ''
    text = raw.strip()
    text = normalize_unicode(text)
    text = _GUILLEMETS.sub('', text)
    text = _EM_DASH.sub('-', text)
    if is_body:
        text = _NOTE_INLINE.sub('', text)
    text = apply_ocr_fixes(text)
    if is_body:
        lines = text.split('\n')
        clean_lines = []
        in_fn = False
        for line in lines:
            s = line.strip()
            if not in_fn and _FOOTNOTE_BLOCK.search(s):
                if not re.match(r'^الفصل\s+\d', s):
                    in_fn = True
            if re.match(r'^(?:الفصل|الكتاب|القسم|الباب|الفرع)\s', s):
                in_fn = False
            if in_fn or is_junk_line(s):
                continue
            clean_lines.append(s)
        text = '\n'.join(clean_lines)
    text = _MULTI_SPACE.sub(' ', text)
    text = _TRAILING_WS.sub('', text)
    text = _LEADING_WS.sub('', text)
    text = _MULTI_NEWLINE.sub('\n\n', text)
    return text.strip()

def clean_context(ctx: str) -> str:
    if not ctx:
        return ''
    parts = [p.strip() for p in ctx.split('>')]
    cleaned = []
    for part in parts:
        part = clean_text(part, is_body=False)
        part = re.sub(r':\s*$', '', part).strip()
        if part and len(part) > 3:
            cleaned.append(part)
    return ' > '.join(cleaned)

_LIST_ITEM = re.compile(
    r'^(\d+|[أبجدهوزحطيكلمنسعفصقرشتثخذضظغ])\s*[-–:]\s+',
    re.UNICODE | re.MULTILINE)

def count_arabic_chars(text: str) -> int:
    return sum(1 for c in text if '\u0600' <= c <= '\u06ff')

def detect_article_references(text: str) -> list:
    pat = re.compile(
        r'(?:الفصل(?:ين|ان)?|الفصول)\s+(\d{1,4}(?:[-\.]\d{1,4})?)'
        r'(?:\s+(?:و|إلى|حتى)\s+(\d{1,4}(?:[-\.]\d{1,4})?))?', re.UNICODE)
    refs = []
    for m in pat.finditer(text):
        refs.append(m.group(1).replace('.', '-'))
        if m.group(2):
            refs.append(m.group(2).replace('.', '-'))
    return sorted(set(refs))

def build_rag_chunk(art_id: str, art_data: dict) -> Optional[dict]:
    contenu = clean_text(art_data.get('contenu', ''), is_body=True)
    if count_arabic_chars(contenu) < MIN_ARABIC_CHARS:
        return None
    contexte   = clean_context(art_data.get('contexte', ''))
    raw_header = art_data.get('header_line', f'الفصل {art_id}')
    header     = clean_text(raw_header, is_body=False) or f'الفصل {art_id}'
    start_p    = art_data.get('starts_on_page')
    end_p      = art_data.get('ends_on_page')
    pages      = (list(range(start_p, end_p + 1))
                  if start_p and end_p and start_p != end_p
                  else ([start_p] if start_p else []))
    arabic_chars = count_arabic_chars(contenu)
    word_count   = len(re.findall(r'[\u0600-\u06ff]+', contenu))
    list_items   = len(_LIST_ITEM.findall(contenu))
    refs         = detect_article_references(contenu)
    rag_parts    = []
    if contexte:
        rag_parts.append(f'السياق: {contexte}')
    rag_parts += [header, contenu]
    return {
        'id':          f'fasl_{art_id.replace("-", "_")}',
        'article':     art_id,
        'header':      header,
        'contexte':    contexte,
        'contenu':     contenu,
        'contenu_rag': '\n'.join(rag_parts),
        'pages':       pages,
        'complete':    art_data.get('complete', True),
        'has_list':    list_items > 0,
        'list_items':  list_items,
        'refs':        refs,
        'word_count':  word_count,
        'char_count':  arabic_chars,
    }


# ════════════════════════════════════════════════════════════════════
# ── PIPELINE CELL 3 ───────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════

def run_cell3_pipeline(ocr_pages_in=None, ocr_cache=OCR_CACHE,
                       struct_out=STRUCT_OUT, rag_out=RAG_OUT) -> tuple:

    if ocr_pages_in is not None:
        ocr_pages = {int(k): v for k, v in ocr_pages_in.items()}
        print(f"  OCR chargé depuis la mémoire : {len(ocr_pages)} pages")
    else:
        with open(ocr_cache, encoding='utf-8') as f:
            ocr_pages = {int(k): v for k, v in json.load(f).items()}
        print(f"  OCR chargé depuis {ocr_cache} : {len(ocr_pages)} pages")

    print("\n" + "━"*60)
    print("ÉTAPE 1/4 — Analyse structurelle page par page")
    print("━"*60)
    pages_analysis = []
    for page_num in sorted(ocr_pages.keys()):
        pa = analyse_page(page_num, ocr_pages[page_num])
        pages_analysis.append(pa)
        print(f"  p{page_num:3d}  ar={pa.arabic_ratio:.2f}"
              f"  arts={len(pa.articles)}"
              f"  structs={len(pa.structural_elements)}"
              f"  notes={len(pa.footnotes)}")

    print("\n" + "━"*60)
    print("ÉTAPE 2/4 — Reconstruction des articles cross-page")
    print("━"*60)
    merged_articles = rebuild_cross_page_articles(pages_analysis)
    incomplete = [aid for aid, a in merged_articles.items() if not a.get("complete")]
    print(f"  Articles totaux            : {len(merged_articles)}")
    print(f"  Potentiellement incomplets : {len(incomplete)}")
    suspicious = {
        aid: art for aid, art in merged_articles.items()
        if (art.get("ends_on_page", 0) - art.get("starts_on_page", 0)) > 5}
    if suspicious:
        print(f"  ⚠️  Fusions > 5 pages :")
        for aid, art in list(suspicious.items())[:10]:
            span = art["ends_on_page"] - art["starts_on_page"]
            print(f"     الفصل {aid} : {art['starts_on_page']} → {art['ends_on_page']} ({span} pages)")
    else:
        print("  Aucune fusion anormale ✓")

    print("\n" + "━"*60)
    print("ÉTAPE 3/4 — Contexte hiérarchique & export structuré")
    print("━"*60)
    art_contexts = build_hierarchy_context(pages_analysis)
    structured   = export_structured(pages_analysis, merged_articles, art_contexts, struct_out)
    print(f"  Contextes construits : {len(art_contexts)}")
    print(f"  Sauvegardé → {struct_out}")

    print("\n" + "━"*60)
    print("ÉTAPE 4/4 — Nettoyage expert & construction chunks RAG")
    print("━"*60)
    articles_raw = structured.get('articles', {})
    chunks  = {}
    skipped = []
    incpl   = []

    for art_id, art_data in sorted(articles_raw.items(), key=lambda x: _sort_key(x[0])):
        chunk = build_rag_chunk(art_id, art_data)
        if chunk is None:
            skipped.append(art_id)
            continue
        chunks[art_id] = chunk
        if not chunk['complete']:
            incpl.append(art_id)

    total_words = sum(c['word_count'] for c in chunks.values())
    total_chars = sum(c['char_count'] for c in chunks.values())
    with_list   = sum(1 for c in chunks.values() if c['has_list'])
    with_refs   = sum(1 for c in chunks.values() if c['refs'])
    avg_words   = total_words / max(len(chunks), 1)

    print(f"  Chunks RAG produits    : {len(chunks)}")
    print(f"  Articles ignorés       : {len(skipped)}")
    if skipped[:10]:
        print(f"    Exemples : {', '.join(skipped[:10])}")
    print(f"  Chunks incomplets      : {len(incpl)}")
    print(f"\n  ── MÉTRIQUES ──────────────────────────────────────")
    print(f"  Mots arabes totaux     : {total_words:,}")
    print(f"  Moy. mots/article      : {avg_words:.0f}")
    print(f"  Articles avec liste    : {with_list} ({100*with_list//max(len(chunks),1)}%)")
    print(f"  Articles avec refs     : {with_refs} ({100*with_refs//max(len(chunks),1)}%)")

    missing_1_65 = [str(i) for i in range(1, 66) if str(i) not in chunks]
    print(f"\n  Manquants parmi 1-65   : {', '.join(missing_1_65) if missing_1_65 else 'aucun ✓'}")
    if '2' in chunks:
        items_2 = chunks['2']['list_items']
        print(f"  الفصل 2 items liste   : {items_2}/4 {'✓' if items_2 == 4 else '⚠️'}")
    if '3' in chunks:
        c3   = chunks['3']
        span = (c3['pages'][-1] - c3['pages'][0]) if len(c3['pages']) > 1 else 0
        flag = '✓' if span <= 3 else f'⚠️  FUSION ({span} pages)'
        print(f"  الفصل 3 span pages    : {span}  {flag}")

    rag_output = {
        'metadata': {
            'source':       struct_out,
            'total_chunks': len(chunks),
            'total_words':  total_words,
            'total_chars':  total_chars,
            'avg_words':    round(avg_words, 1),
            'incomplete':   len(incpl),
            'with_lists':   with_list,
            'with_refs':    with_refs,
        },
        'chunks': chunks,
    }
    with open(rag_out, 'w', encoding='utf-8') as f:
        json.dump(rag_output, f, ensure_ascii=False, indent=2)
    print(f"\n  Sauvegardé → {rag_out}")

    return structured, rag_output


# ════════════════════════════════════════════════════════════════════
# ── EXÉCUTION CELL 3 ──────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════

print("━"*60)
print("CELL 3 — Analyse structurelle + Nettoyage RAG  (v5)")
print("━"*60)

try:
    _ = ocr_pages   # type: ignore
    print("  Variable `ocr_pages` trouvée en scope")
    structured, rag_result = run_cell3_pipeline(ocr_pages_in=ocr_pages)
except NameError:
    print(f"  `ocr_pages` absent — chargement depuis {OCR_CACHE}")
    structured, rag_result = run_cell3_pipeline()

chunks = rag_result['chunks']

print(f"\n{'═'*60}\nAPERÇU — 3 PREMIERS CHUNKS\n{'═'*60}")
for art_id, chunk in list(chunks.items())[:3]:
    print(f"\n── الفصل {art_id} ──")
    print(f"   Pages    : {chunk['pages']}")
    print(f"   Complet  : {chunk['complete']} | Mots: {chunk['word_count']}")
    print(f"   Contenu  :\n{chunk['contenu'][:300]}\n")

print(f"\nExport structuré → {STRUCT_OUT}")
print(f"Chunks RAG       → {RAG_OUT}  ({rag_result['metadata']['total_chunks']} chunks)")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELL 3 — Analyse structurelle + Nettoyage RAG  (v5)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Variable `ocr_pages` trouvée en scope
  OCR chargé depuis la mémoire : 268 pages

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ÉTAPE 1/4 — Analyse structurelle page par page
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  p  1  ar=0.83  arts=0  structs=0  notes=0
  p  2  ar=0.83  arts=2  structs=0  notes=0
  p  3  ar=0.98  arts=0  structs=1  notes=0
  p  4  ar=1.00  arts=0  structs=0  notes=0
  p  5  ar=0.96  arts=2  structs=3  notes=0
  p  6  ar=0.92  arts=3  structs=4  notes=0
  p  7  ar=0.96  arts=4  structs=0  notes=0
  p  8  ar=0.98  arts=5  structs=0  notes=0
  p  9  ar=0.97  arts=6  structs=1  notes=0
  p 10  ar=0.80  arts=5  structs=0  notes=0
  p 11  ar=0.95  arts=5  structs=0  notes=0
  p 12  ar=0.97  arts=8  structs=0  notes=0
  p 13  ar=0.96  arts=6  structs=0  notes=0
 

In [31]:


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 (v5) — Parsing des articles depuis structured + chunks
#
# Ne relit PAS l'OCR brut. Consomme directement :
#   structured['articles']  → texte structuré + contexte hiérarchique
#   chunks                  → texte nettoyé RAG (priorité)
#
# Le cache périmé est supprimé au démarrage pour forcer un re-parse propre.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "━"*60)
print("CELL 4 — Parsing des articles (v5 — depuis structured + chunks)")
print("━"*60)

# ── Suppression du cache périmé ───────────────────────────────────
if os.path.exists(ARTICLES_CACHE):
    os.remove(ARTICLES_CACHE)
    print(f"  Cache périmé supprimé : {ARTICLES_CACHE}")

# ─────────────────────────────────────────────────────────────────
# DATACLASS
# ─────────────────────────────────────────────────────────────────

@dataclass
class LegalArticle:
    article_num:     str
    article_num_int: int
    text:            str
    book:            str  = ""
    section:         str  = ""
    subsection:      str  = ""
    char_count:      int  = 0
    contexte:        str  = ""
    pages:           list = None
    complete:        bool = True
    refs:            list = None

    def __post_init__(self):
        if self.pages is None: self.pages = []
        if self.refs  is None: self.refs  = []

    def to_display(self):
        lines = [f"{'='*60}", f"الفصل {self.article_num}"]
        if self.book:     lines.append(f"[{self.book}]")
        if self.section:  lines.append(f"[{self.section}]")
        if self.contexte: lines.append(f"[السياق: {self.contexte[:80]}]")
        lines += ["", self.text, ""]
        return "\n".join(lines)

    def to_chunk_text(self):
        hdr = f"[الفصل {self.article_num}]"
        if self.book:    hdr += f" [{self.book}]"
        if self.section: hdr += f" [{self.section}]"
        return f"{hdr}\n{self.text}"

# ─────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────

def _num_to_int(s: str) -> int:
    s = s.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    m = re.search(r"\d+", s)
    return int(m.group()) if m else 0

def _extract_hierarchy(contexte: str) -> tuple:
    book = sect = subs = ""
    if not contexte:
        return book, sect, subs
    for part in contexte.split('>'):
        part = part.strip()
        if re.match(r'^الكتاب', part):
            book = part
        elif re.match(r'^(?:القسم|الباب)', part):
            sect = part
        elif re.match(r'^الفرع', part):
            subs = part
    return book, sect, subs

# ── Nettoyage corps v5 ────────────────────────────────────────────
_JUNK_BODY_PAT = re.compile(
    r'(?:المملكة\s+المغربية|وزارة\s+(?:العدل|العدار|العدد)'
    r'|مديرية\s+التشريع|قانون\s+الالتزامات\s+والعقود|صيغة\s+محينة'
    r'|---\s*PAGE\s+\d+)', re.UNICODE)

def _clean_body_v5(text: str) -> str:
    lines = text.split('\n')
    out   = []
    for line in lines:
        s = line.strip()
        if _JUNK_BODY_PAT.search(s):
            continue
        if _PAGE_NUMBER_PAT.match(s):
            continue
        out.append(line)
    text = '\n'.join(out)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# ─────────────────────────────────────────────────────────────────
# CONSTRUCTION DES ARTICLES
# ─────────────────────────────────────────────────────────────────

def build_articles_from_structured(structured: dict, chunks: dict) -> list:
    articles_raw  = structured.get('articles', {})
    result        = []
    n_from_chunk  = 0
    n_from_struct = 0
    n_skipped     = 0

    for art_id, art_data in sorted(articles_raw.items(), key=lambda x: _sort_key(x[0])):
        num_int = _num_to_int(art_id.split('-')[0])
        if num_int == 0 or num_int > LOC_MAX_NUM:
            n_skipped += 1
            continue

        chunk = chunks.get(art_id)
        if chunk and chunk.get('contenu', '').strip():
            raw_text = chunk['contenu']
            contexte = chunk.get('contexte', '')
            pages    = chunk.get('pages', [])
            complete = chunk.get('complete', True)
            refs     = chunk.get('refs', [])
            n_from_chunk += 1
        else:
            raw_text = art_data.get('contenu', '')
            contexte = art_data.get('contexte', '')
            pages_s  = art_data.get('starts_on_page')
            pages_e  = art_data.get('ends_on_page')
            pages    = (list(range(pages_s, pages_e + 1))
                        if pages_s and pages_e and pages_s != pages_e
                        else ([pages_s] if pages_s else []))
            complete = art_data.get('complete', True)
            refs     = art_data.get('internal_refs', [])
            n_from_struct += 1

        text = _clean_body_v5(raw_text)
        if len(text) < 20:
            n_skipped += 1
            continue

        book, sect, subs = _extract_hierarchy(contexte)

        result.append(LegalArticle(
            article_num=art_id,
            article_num_int=num_int,
            text=text,
            book=book,
            section=sect,
            subsection=subs,
            char_count=len(text),
            contexte=contexte,
            pages=pages,
            complete=complete,
            refs=refs,
        ))

    print(f"  Depuis chunks    : {n_from_chunk}")
    print(f"  Depuis structured: {n_from_struct}")
    print(f"  Ignorés          : {n_skipped}")
    return result

# ─────────────────────────────────────────────────────────────────
# EXÉCUTION CELL 4
# ─────────────────────────────────────────────────────────────────

print(f"\n  Consommation de structured ({len(structured.get('articles', {}))} articles)"
      f" + chunks ({len(chunks)} chunks)")

articles = build_articles_from_structured(structured, chunks)

if not articles:
    raise RuntimeError("Aucun article extrait.")

with open(ARTICLES_CACHE, 'w', encoding='utf-8') as f:
    json.dump([asdict(a) for a in articles], f, ensure_ascii=False, indent=2)
print(f"\n  {len(articles)} articles sauvegardés → {ARTICLES_CACHE}")

# ── Stats ─────────────────────────────────────────────────────────
chars = [a.char_count for a in articles]
print(f"\n  Total articles : {len(articles)}")
print(f"  Plage          : الفصل {articles[0].article_num} → الفصل {articles[-1].article_num}")
print(f"  Chars/article  : min={min(chars)} | moy={sum(chars)//len(chars)} | max={max(chars)}")

# ── Vérification الفصل 2 ─────────────────────────────────────────
art2 = next((a for a in articles if a.article_num == "2"), None)
if art2:
    print(f"\n  VÉRIFICATION الفصل 2 :")
    # Le texte ne doit PAS contenir de header institutionnel
    has_inst = bool(_JUNK_BODY_PAT.search(art2.text))
    print(f"    Header institutionnel dans le corps : {'⚠️  OUI' if has_inst else 'non ✓'}")
    print(f"    Texte[:300] :\n{art2.text[:300]}")
    for item in ["1 -", "2 -", "3 -", "4 -"]:
        print(f"    Item {item} : {'PRÉSENT ✓' if item in art2.text else 'MANQUANT ⚠️'}")
    if art2.contexte: print(f"    Contexte : {art2.contexte[:80]}")
    if art2.pages:    print(f"    Pages    : {art2.pages}")

# ── Vérification الفصل 3 ─────────────────────────────────────────
art3 = next((a for a in articles if a.article_num == "3"), None)
if art3:
    span = (art3.pages[-1] - art3.pages[0]) if len(art3.pages) > 1 else 0
    flag = '✓' if span <= 3 else f'⚠️  FUSION ({span} pages)'
    print(f"\n  VÉRIFICATION الفصل 3 : pages={art3.pages[:5]} span={span} {flag}")
    print(f"    Chars={art3.char_count} | Texte[:200] :\n    {art3.text[:200]}")

print(f"\n  Cache articles → {ARTICLES_CACHE}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELL 4 — Parsing des articles (v5 — depuis structured + chunks)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Cache périmé supprimé : /content/loc_articles.json

  Consommation de structured (1231 articles) + chunks (1202 chunks)
  Depuis chunks    : 1202
  Depuis structured: 29
  Ignorés          : 29

  1202 articles sauvegardés → /content/loc_articles.json

  Total articles : 1202
  Plage          : الفصل 1 → الفصل 758 مكرر
  Chars/article  : min=20 | moy=223 | max=1969

  VÉRIFICATION الفصل 2 :
    Header institutionnel dans le corps : non ✓
    Texte[:300] :
الأركان 9 اللازمة لصحة الالتزامات الناشئة عن التعبير عن الإرادة هي:
1 - الأهلية للالتزام؛
2 - تعبير صحيح عن الإرادة يقع علي العناصر الأساسية للالتزام؛
3 - شيء محقق يصلح لأن يكون محلا للالتزام؛
4 - سبب مشروع للالتزام.
    Item 1 - : PRÉSENT ✓
    Item 2 - : PRÉSENT ✓
    Item 3 - : PRÉSENT ✓
    Item 4 - : PRÉSENT ✓
    Contexte : الكتاب الأول: الالت

In [32]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 (v5) — Embeddings + Index FAISS
#
# Adapté pour consommer directement les chunks RAG (dict) issus de cell 3/4 v5.
# Utilise chunk['contenu_rag'] pour l'embedding (contexte + header + corps nettoyé).
# Fallback sur chunk['contenu'] si contenu_rag absent.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "━"*60)
print("CELL 5 — Embeddings + Index FAISS  (v5 — depuis chunks RAG)")
print("━"*60)

import torch, numpy as np, faiss
from sentence_transformers import SentenceTransformer

EMBED_CACHE = "/content/loc_embeddings.npy"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device : {DEVICE}")

# ─────────────────────────────────────────────────────────────────
# SOURCE DES CHUNKS
# ─────────────────────────────────────────────────────────────────
# Priorité : variable `chunks` en mémoire (cell 3/4 v5)
# Fallback  : fichier RAG_OUT sur disque

try:
    _ = chunks   # type: ignore
    print(f"  Variable `chunks` trouvée en mémoire : {len(chunks)} chunks")
except NameError:
    print(f"  `chunks` absent — chargement depuis {RAG_OUT}")
    with open(RAG_OUT, encoding='utf-8') as f:
        rag_result = json.load(f)
    chunks = rag_result['chunks']
    print(f"  Chargé : {len(chunks)} chunks")

# ─────────────────────────────────────────────────────────────────
# CONSTRUCTION DES TEXTES À ENCODER
# ─────────────────────────────────────────────────────────────────
# On trie par clé numérique pour stabiliser l'ordre des embeddings.

sorted_ids    = sorted(chunks.keys(), key=_sort_key)
chunk_ids     = sorted_ids                                # liste ordonnée des IDs
chunk_texts   = [
    chunks[cid].get('contenu_rag') or chunks[cid].get('contenu', '')
    for cid in chunk_ids
]

print(f"  Chunks à encoder : {len(chunk_texts)}")
print(f"  Exemple contenu_rag (الفصل {chunk_ids[0]}) :\n"
      f"    {chunk_texts[0][:200]!r}\n")

# ─────────────────────────────────────────────────────────────────
# CHARGEMENT DU MODÈLE D'EMBEDDING
# ─────────────────────────────────────────────────────────────────

EMBED_MODELS = [
    "CAMeL-Lab/camel-bert-base-sts",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

embed_model = None
for mid in EMBED_MODELS:
    try:
        print(f"  Chargement : {mid}…")
        embed_model = SentenceTransformer(mid, device=DEVICE)
        print(f"  ✓ Modèle chargé : {mid}")
        break
    except Exception as e:
        print(f"  ✗ Échec : {str(e)[:80]}")

if embed_model is None:
    raise RuntimeError("Aucun modèle d'embedding disponible.")

# ─────────────────────────────────────────────────────────────────
# EMBEDDINGS (cache invalidé si taille change)
# ─────────────────────────────────────────────────────────────────

embeddings = None
if os.path.exists(EMBED_CACHE):
    embeddings = np.load(EMBED_CACHE)
    if embeddings.shape[0] != len(chunk_texts):
        print(f"  Cache périmé ({embeddings.shape[0]} ≠ {len(chunk_texts)}) → recalcul")
        os.remove(EMBED_CACHE)
        embeddings = None
    else:
        print(f"  Cache valide chargé : {embeddings.shape}")

if embeddings is None:
    print(f"  Encodage de {len(chunk_texts)} textes…")
    embeddings = embed_model.encode(
        chunk_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    np.save(EMBED_CACHE, embeddings)
    print(f"  Embeddings sauvegardés → {EMBED_CACHE}  shape={embeddings.shape}")

# ─────────────────────────────────────────────────────────────────
# INDEX FAISS
# ─────────────────────────────────────────────────────────────────

DIM       = embeddings.shape[1]
cpu_index = faiss.IndexFlatIP(DIM)
cpu_index.add(embeddings.astype(np.float32))

if DEVICE == "cuda":
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    print(f"  FAISS GPU : {index.ntotal} vecteurs × {DIM} dims")
else:
    index = cpu_index
    print(f"  FAISS CPU : {index.ntotal} vecteurs × {DIM} dims")

# ─────────────────────────────────────────────────────────────────
# FONCTION DE RECHERCHE
# ─────────────────────────────────────────────────────────────────

def search_articles(query: str, top_k: int = 5) -> list:
    """
    Retourne les top_k chunks les plus proches de la requête.
    Chaque résultat est un dict avec : article, score, contenu, contexte, pages.
    """
    q_vec = embed_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)
    scores, indices = index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(chunk_ids):
            continue
        cid   = chunk_ids[idx]
        chunk = chunks[cid]
        results.append({
            "article":  chunk.get("article", cid),
            "score":    float(score),
            "contenu":  chunk.get("contenu", ""),
            "contexte": chunk.get("contexte", ""),
            "pages":    chunk.get("pages", []),
            "complete": chunk.get("complete", True),
        })
    return results

# ─────────────────────────────────────────────────────────────────
# VÉRIFICATION RAPIDE
# ─────────────────────────────────────────────────────────────────

print(f"\n  Test de recherche : 'الأهلية المدنية'")
for r in search_articles("الأهلية المدنية", top_k=3):
    print(f"    الفصل {r['article']:>6}  score={r['score']:.4f}"
          f"  pages={r['pages']}  {'✓' if r['complete'] else '⚠️'}")
    print(f"      {r['contenu'][:120]}")

print(f"\n  chunk_ids[0..4] : {chunk_ids[:5]}")
print(f"  Index prêt      : {index.ntotal} vecteurs")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELL 5 — Embeddings + Index FAISS  (v5 — depuis chunks RAG)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Device : cuda
  Variable `chunks` trouvée en mémoire : 1202 chunks
  Chunks à encoder : 1202
  Exemple contenu_rag (الفصل 1) :
    'السياق: الكتاب الأول من الظهير الشريف المعتبر بمثابة قانون الالتزامات والعقود، وذلك بمقتضي المادة 3 من القانون رقم 53.05 يتعلق بالتبادل الإلكتروني للمعطيات القانونية. > الباب الثالث: الالتزامات الناشئ'

  Chargement : CAMeL-Lab/camel-bert-base-sts…
  ✗ Échec : CAMeL-Lab/camel-bert-base-sts is not a local folder and is not a valid model ide
  Chargement : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2…


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✓ Modèle chargé : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  Encodage de 1202 textes…


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

  Embeddings sauvegardés → /content/loc_embeddings.npy  shape=(1202, 384)
  FAISS GPU : 1202 vecteurs × 384 dims

  Test de recherche : 'الأهلية المدنية'
    الفصل     95  score=0.4202  pages=[28]  ✓
      لا محل للمسؤولية المدنية في حالة الدفاع الشرعي، أو إذا كان الضرر قد نتج عن حادث فجائي أو قوة قاهرة لم يسبقها أو يصطحبها 
    الفصل     94  score=0.4180  pages=[28]  ✓
      لا محل للمسؤولية المدنية، إذا فعل شخص بغير قصد الإضرار ما كان له الحق في فعله.
غير أنه إذا كان من شأن مباشرة هذا الحق أن
    الفصل      3  score=0.4124  pages=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54]  ✓
      الأهلية المدنية للفرد تخضع لقانون أحواله الشخصية 14. وكل شخص أهل للإلزام والالتزام 15 ما لم يصرح قانون أحواله الشخصية بغ

  chunk_ids[0..4] : ['1', '2', '3', '4', '5']
  Index prêt      : 1202 vecteurs


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 (v5) — BM25 + HybridRetriever AVEC SEUIL RRF
#
# Adapté pour consommer les chunks RAG (dicts) au lieu des objets LegalArticle.
# Utilise chunk['contenu_rag'] pour le BM25 (même texte que l'embedding).
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "━"*60)
print("CELL 6 — BM25 + HybridRetriever  (v5 — depuis chunks RAG)")
print("━"*60)

from rank_bm25 import BM25Okapi
from dataclasses import dataclass as _dc

STOP_WORDS = {
    "من", "في", "على", "عن", "إلى", "هذا", "هذه", "التي", "الذي", "وأن",
    "أن", "لا", "ما", "مع", "أو", "ولا", "لم", "كان", "يكون", "ذلك",
    "هو", "هي", "إذا", "إذ", "عند", "بعد", "قبل", "حتى", "إن", "كل",
    "غير", "بين", "كما", "حين", "منذ", "لكن", "ثم", "لدى",
}

# Score RRF maximum théorique (rang 1 dans dense ET sparse) = 1/(60+1)*2 ≈ 0.0328
# Seuil à ~37% du max → filtre les faux positifs.
RRF_MIN_THRESHOLD = 0.012

# ─────────────────────────────────────────────────────────────────
# SOURCE DES CHUNKS (même fallback que cell 5)
# ─────────────────────────────────────────────────────────────────

try:
    _ = chunks   # type: ignore
    print(f"  Variable `chunks` trouvée en mémoire : {len(chunks)} chunks")
except NameError:
    print(f"  `chunks` absent — chargement depuis {RAG_OUT}")
    with open(RAG_OUT, encoding='utf-8') as f:
        rag_result = json.load(f)
    chunks = rag_result['chunks']

try:
    _ = chunk_ids   # type: ignore   # liste ordonnée produite par cell 5
    print(f"  `chunk_ids` trouvé en mémoire : {len(chunk_ids)} IDs")
except NameError:
    chunk_ids = sorted(chunks.keys(), key=_sort_key)
    print(f"  `chunk_ids` reconstruit : {len(chunk_ids)} IDs")

# ─────────────────────────────────────────────────────────────────
# TOKENISATION ARABE
# ─────────────────────────────────────────────────────────────────

def tokenize_ar(text: str) -> list:
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[^\u0600-\u06FF\s\d]", " ", text)
    return [t for t in text.split() if len(t) > 2 and t not in STOP_WORDS]

# ─────────────────────────────────────────────────────────────────
# INDEX BM25 (aligné sur chunk_ids → même ordre que FAISS)
# ─────────────────────────────────────────────────────────────────
# Utilise contenu_rag (contexte + header + corps) comme l'embedding cell 5.
# Fallback sur contenu si contenu_rag absent.

bm25_texts = [
    chunks[cid].get('contenu_rag') or chunks[cid].get('contenu', '')
    for cid in chunk_ids
]
tokenized = [tokenize_ar(t) for t in bm25_texts]
bm25      = BM25Okapi(tokenized)

print(f"  BM25 : {len(tokenized)} documents indexés")

# ─────────────────────────────────────────────────────────────────
# DATACLASS RÉSULTAT
# ─────────────────────────────────────────────────────────────────

@_dc
class RetrievedChunk:
    chunk_id:    str           # ex. "45", "12-1"
    chunk:       dict          # le dict RAG complet
    dense_score: float
    bm25_score:  float
    rrf_score:   float
    dense_rank:  int
    bm25_rank:   int

    @property
    def article_num(self):
        return self.chunk.get('article', self.chunk_id)

    @property
    def text_preview(self):
        return self.chunk.get('contenu', '')[:80]

# ─────────────────────────────────────────────────────────────────
# HYBRID RETRIEVER
# ─────────────────────────────────────────────────────────────────

class HybridRetriever:
    """
    Retrieval hybride Dense (FAISS) + BM25 + RRF avec filtrage de pertinence.

    Les indices FAISS et BM25 sont alignés sur chunk_ids.

    Méthodes :
      retrieve(query, top_k)          → top-K sans seuil  (debug)
      retrieve_filtered(query, top_k) → top-K avec seuil  (production)
    """

    def __init__(self, cids, chunks_dict, faiss_idx, emb_model, bm25_model, k=60):
        self.cids  = cids           # liste ordonnée des IDs (align FAISS ↔ BM25)
        self.data  = chunks_dict    # dict {id: chunk}
        self.index = faiss_idx
        self.embed = emb_model
        self.bm25  = bm25_model
        self.k     = k

    # ── Dense ─────────────────────────────────────────────────────
    def _dense(self, query: str, n: int = 20) -> list:
        vec = self.embed.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True
        ).astype(np.float32)
        scores, ids = self.index.search(vec, n)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    # ── Sparse ────────────────────────────────────────────────────
    def _sparse(self, query: str, n: int = 20) -> list:
        tokens = tokenize_ar(query)
        if not tokens:
            return []
        sc  = self.bm25.get_scores(tokens)
        top = np.argsort(sc)[::-1][:n]
        return [(int(i), float(sc[i])) for i in top]

    # ── RRF ───────────────────────────────────────────────────────
    def _rrf(self, dense_res: list, sparse_res: list, n: int = 10) -> list:
        rrf_sc = {}
        d_rank, b_rank, d_sc, b_sc = {}, {}, {}, {}

        for rk, (i, s) in enumerate(dense_res):
            rrf_sc[i]  = rrf_sc.get(i, 0) + 1.0 / (self.k + rk + 1)
            d_rank[i] = rk + 1
            d_sc[i]   = s

        for rk, (i, s) in enumerate(sparse_res):
            rrf_sc[i]  = rrf_sc.get(i, 0) + 1.0 / (self.k + rk + 1)
            b_rank[i] = rk + 1
            b_sc[i]   = s

        top = sorted(rrf_sc, key=rrf_sc.get, reverse=True)[:n]
        results = []
        for i in top:
            if not (0 <= i < len(self.cids)):
                continue
            cid = self.cids[i]
            if cid not in self.data:
                continue
            results.append(RetrievedChunk(
                chunk_id=cid,
                chunk=self.data[cid],
                dense_score=d_sc.get(i, 0.0),
                bm25_score=b_sc.get(i, 0.0),
                rrf_score=rrf_sc[i],
                dense_rank=d_rank.get(i, 999),
                bm25_rank=b_rank.get(i, 999),
            ))
        return results

    # ── API publique ──────────────────────────────────────────────
    def retrieve(self, query: str, top_k: int = 5) -> list:
        """Sans seuil — usage debug."""
        return self._rrf(self._dense(query, 20), self._sparse(query, 20), top_k)

    def retrieve_filtered(self, query: str, top_k: int = 5,
                          min_threshold: float = RRF_MIN_THRESHOLD) -> list:
        """
        Production : retourne uniquement les chunks avec RRF ≥ seuil.
        Fallback top-3 si aucun candidat ne passe le seuil.
        """
        candidates = self._rrf(
            self._dense(query, 30),
            self._sparse(query, 30),
            top_k * 2,
        )
        filtered = [r for r in candidates if r.rrf_score >= min_threshold]
        if not filtered:
            print(f"   Aucun chunk au-dessus du seuil {min_threshold:.4f} — fallback top-3")
            return candidates[:3]
        return filtered[:top_k]

# ─────────────────────────────────────────────────────────────────
# INSTANCIATION
# ─────────────────────────────────────────────────────────────────

retriever = HybridRetriever(
    cids=chunk_ids,
    chunks_dict=chunks,
    faiss_idx=index,
    emb_model=embed_model,
    bm25_model=bm25,
)

# ─────────────────────────────────────────────────────────────────
# TEST
# ─────────────────────────────────────────────────────────────────

print("\n  Test retrieve_filtered : 'مدة العقد غير محددة عقد العمل'")
for r in retriever.retrieve_filtered("مدة العقد غير محددة عقد العمل", top_k=6):
    flag = '✓' if r.chunk.get('complete', True) else '⚠️'
    print(f"  الفصل {r.article_num:>6} | RRF={r.rrf_score:.4f}"
          f" | dense={r.dense_rank} bm25={r.bm25_rank}"
          f" | {flag} | {r.text_preview}…")

print(f"\n  Retriever prêt — {len(chunk_ids)} chunks indexés")

BM25 : 17 documents indexés
Index direct : 25 entrées

Test retrieval filtre : 'مدة العقد غير محددة عقد العمل'
  الفصل    443 | RRF=0.0320 | الاتفاقات وغيرها من الأفعال القانونية التي يكون من شأنه…
  الفصل    233 | RRF=0.0318 | الإيجاب الموجه لشخص حاضرء من غير تحديد ميعادء يعتبر كأن…
  الفصل   7-65 | RRF=0.0310 | عندما بطلب الإدلاء بعدة أصول» تعتبر هذه الإلزامية مستوف…
  الفصل   4-65 | RRF=0.0308 | عن . . ١ل‏ .م اه لحي 5 ‎٠.‏ م 57 الام

يتعين على كل من …
  الفصل    425 | RRF=0.0301 | المحررات العرفية دليل على تاريخها بين المتعاقدين وورثته…
  الفصل   5-65 | RRF=0.0299 | يشترط لصحة إبرام العقد أن يكون من أرسل العرض إليه قد تم…


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — LLM Qwen2.5-7B-Instruct (inchangée)
# ══════════════════════════════════════════════════════════════════════════════
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

MODEL_REPO = "bartowski/Qwen2.5-7B-Instruct-GGUF"
MODEL_FILE = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
MODEL_PATH = f"/content/{MODEL_FILE}"

if not os.path.exists(MODEL_PATH):
    print(f"Téléchargement {MODEL_FILE} (~4.4 GB)…")
    hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir="/content")
else:
    print(f"Modèle déjà présent : {MODEL_PATH}")

llm = Llama(
    model_path=MODEL_PATH, n_gpu_layers=-1, n_ctx=4096,
    n_batch=512, verbose=False, seed=42, chat_format="chatml",
)
print("Qwen2.5-7B-Instruct chargé")

_t = llm.create_chat_completion(
    messages=[{"role": "user", "content": "قل: جاهز"}], max_tokens=10
)
print(f"Test LLM : {_t['choices'][0]['message']['content']}")

Téléchargement Qwen2.5-7B-Instruct-Q4_K_M.gguf (~4.4 GB)…


Qwen2.5-7B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Qwen2.5-7B-Instruct chargé
Test LLM : جاهز


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 (v6) — Agent juridique JURISTE
#
# Architecture multi-étapes :
#   ÉTAPE 0 — Analyse de la question (type, domaine, ambiguïté)
#   ÉTAPE 1 — Lookup direct + retrieval hybride étendu
#   ÉTAPE 2 — Re-ranking LLM : juge quels chunks sont vraiment utiles
#   ÉTAPE 3 — Raisonnement juridique en chaîne (chain-of-thought)
#   ÉTAPE 4 — Réponse finale structurée (juriste → client)
#   ÉTAPE 5 — Auto-critique : détecte les failles dans sa propre réponse
#   ÉTAPE 6 — Affichage ciblé des textes officiels + méta-rapport
#
# Capacités nouvelles vs v5 :
#   • Détecte les questions multi-domaines et les traite séparément
#   • Re-ranking des chunks avant injection (élimine les faux positifs résiduels)
#   • Chain-of-thought juridique explicite (syllogisme : règle → faits → conclusion)
#   • Auto-critique post-génération (contradictions, lacunes, incertitudes)
#   • Deux formats de sortie : réponse client (claire) + analyse juriste (technique)
#   • Gestion des questions sans réponse dans le LOC (hors-champ explicite)
#   • Traçabilité complète : chaque affirmation est liée à un فصل précis
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "━"*60)
print("CELL 8 — Agent juridique JURISTE  (v6)")
print("━"*60)

import json as _json
from dataclasses import dataclass as _dc, field as _field
from typing import Optional

# ─────────────────────────────────────────────────────────────────
# DATACLASSES DE RÉSULTAT
# ─────────────────────────────────────────────────────────────────

@_dc
class JuristAnalysis:
    question:          str
    question_type:     str        # "factuelle" | "interprétative" | "comparative" | "procédurale"
    domains:           list       # ex. ["عقد العمل", "الفسخ"]
    ambiguities:       list       # ambiguïtés détectées dans la question
    chunks_retrieved:  list       # tous les chunks candidats
    chunks_selected:   list       # après re-ranking LLM
    chain_of_thought:  str        # raisonnement intermédiaire
    answer_client:     str        # réponse claire pour le client
    answer_juriste:    str        # analyse technique complète
    self_critique:     str        # lacunes / incertitudes détectées
    cited_articles:    list       # [{id, extrait, pertinence}]
    out_of_scope:      bool       # True si la question dépasse le LOC
    confidence:        str        # "haute" | "moyenne" | "faible"
    processing_log:    list       # trace de traitement

# ─────────────────────────────────────────────────────────────────
# PROMPTS SPÉCIALISÉS
# ─────────────────────────────────────────────────────────────────

_PROMPT_ANALYSE = """\
أنت محلل قانوني متخصص في قانون الالتزامات والعقود المغربي (LOC).

مهمتك: تحليل السؤال التالي وإرجاع JSON فقط (بدون أي نص إضافي).

السؤال: {question}

أرجع هذا الـ JSON حصرا:
{{
  "type": "factuelle|interprétative|comparative|procédurale",
  "domaines": ["domain1", "domain2"],
  "ambiguites": ["ambiguité1"],
  "mots_cles_arabes": ["mot1", "mot2"],
  "hors_loc": false
}}

- type "factuelle": سؤال عن نص محدد أو حكم واضح
- type "interprétative": يتطلب تفسير نص غامض أو تطبيق على حالة
- type "comparative": مقارنة بين وضعيتين قانونيتين
- type "procédurale": سؤال عن إجراءات أو مسطرة
- hors_loc: true إذا كان السؤال خارج نطاق قانون الالتزامات والعقود
"""

_PROMPT_RERANKING = """\
أنت قاضٍ يختار الفصول القانونية ذات الصلة بسؤال محدد.

السؤال: {question}
نوع السؤال: {question_type}
المجالات: {domains}

الفصول المتاحة:
{chunks_list}

مهمتك: لكل فصل، قيّم صلته بالسؤال من 0 إلى 10، وأرجع JSON فقط:
{{
  "selected": [
    {{"id": "45", "score": 9, "raison": "ينص صراحة على..."}},
    {{"id": "12", "score": 7, "raison": "يتعلق بـ..."}}
  ],
  "rejected": [
    {{"id": "958", "raison": "يتعلق بالوكالة لا بعقد العمل"}}
  ]
}}

اختر فقط الفصول ذات الصلة المباشرة (score ≥ 5).
"""

_PROMPT_COT = """\
أنت فقيه قانوني مغربي متخصص في قانون الالتزامات والعقود.

السؤال: {question}
نوع السؤال: {question_type}

الفصول المختارة:
{context}

قم بالتحليل القانوني وفق المنهج الآتي (سلسلة الاستدلال):

1. **تحديد القاعدة القانونية** : ما هي النصوص الواجبة التطبيق؟
2. **تكييف الوقائع** : كيف تنطبق هذه النصوص على السؤال؟
3. **الاستنتاج القانوني** : ما هو الحكم القانوني؟
4. **التحفظات** : هل ثمة استثناءات أو حالات خاصة؟
5. **الفراغات** : هل يوجد غموض أو نقص في النصوص المتاحة؟

أرجع JSON فقط:
{{
  "regle_juridique": "النص القانوني المنطبق...",
  "qualification": "التكييف القانوني للوقائع...",
  "conclusion": "الحكم القانوني...",
  "reserves": ["تحفظ1", "تحفظ2"],
  "lacunes": ["فراغ1"],
  "articles_cles": ["45", "12-1"]
}}
"""

_PROMPT_REPONSE_CLIENT = """\
أنت محامٍ يشرح حكماً قانونياً لموكله بلغة واضحة ومفهومة.

السؤال: {question}
التحليل القانوني: {cot_json}
الفصول المستعملة: {articles_used}

اكتب إجابة واضحة للموكل:
- ابدأ بالجواب المباشر (نعم/لا/يتوقف على...)
- اشرح الحكم بلغة بسيطة مع ذكر الفصل القانوني
- نبّه على الاستثناءات المهمة
- اختم بتوصية عملية

الإجابة يجب أن تكون بالعربية، واضحة، بين 3 و6 فقرات.
"""

_PROMPT_REPONSE_JURISTE = """\
أنت فقيه قانوني يكتب مذكرة قانونية تقنية.

السؤال: {question}
التحليل: {cot_json}
الفصول: {context}

اكتب تحليلاً قانونياً تقنياً يتضمن:
1. **التأطير القانوني** : الفصول المنطبقة مع اقتباس الأجزاء الحاسمة
2. **التكييف** : التوصيف القانوني الدقيق للمسألة
3. **الحكم** : الحل القانوني مع التعليل
4. **المقارنة** (إن وجدت) : الحالات المشابهة في النصوص
5. **التحفظات الفنية** : الغموض، التعارض، الفراغات
6. **المراجع** : الفصول بالترتيب مع أهميتها

الإجابة يجب أن تكون بالعربية، دقيقة، تقنية.
"""

_PROMPT_AUTOCRITIQUE = """\
أنت ناقد قانوني يراجع إجابة زميله.

السؤال الأصلي: {question}
الإجابة المعطاة: {answer}
الفصول المستعملة: {articles_used}

حدد بدقة:
1. هل الإجابة مكتملة أم تجاهلت جانباً مهماً؟
2. هل هناك تفسيرات بديلة ممكنة للنصوص؟
3. هل هناك فصول كان يجب استحضارها ولم تُستحضر؟
4. هل الاستنتاج يتناقض مع أي نص من النصوص المقدمة؟
5. ما مستوى اليقين القانوني للإجابة؟

أرجع JSON فقط:
{{
  "completude": "complète|partielle|insuffisante",
  "interpretations_alternatives": ["interp1"],
  "articles_manquants": ["45"],
  "contradictions": ["contradiction1"],
  "niveau_certitude": "haute|moyenne|faible",
  "avertissements": ["mise en garde 1"]
}}
"""

# ─────────────────────────────────────────────────────────────────
# AGENT PRINCIPAL
# ─────────────────────────────────────────────────────────────────

class LawJuristAgent:
    """
    Agent juridique de niveau juriste pour le LOC marocain.

    Architecture en 6 étapes avec raisonnement en chaîne,
    re-ranking LLM et auto-critique post-génération.
    """

    _ART_REF_RE = re.compile(r"الفصل\s*(\d+(?:[.\-]\d+)?)", re.UNICODE)

    def __init__(self, retriever, llm, chunks_dict, chunk_ids,
                 verbose: bool = True):
        self.retriever = retriever
        self.llm       = llm
        self.data      = chunks_dict
        self.cids      = chunk_ids
        self.verbose   = verbose
        self.log       = []

        # Index num → chunk_id
        self._num_idx = {}
        for cid, chunk in chunks_dict.items():
            art = chunk.get('article', cid)
            self._num_idx[art] = cid
            self._num_idx[cid] = cid
            self._num_idx.setdefault(str(_num_to_int(art)), cid)

    # ── Utilitaires ───────────────────────────────────────────────

    def _log(self, msg: str):
        self.log.append(msg)
        if self.verbose:
            print(f"  {msg}")

    def _llm_call(self, prompt: str, max_tokens: int = 800,
                  temperature: float = 0.05) -> str:
        resp = self.llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
            repeat_penalty=1.1,
        )
        return resp["choices"][0]["message"]["content"].strip()

    def _parse_json(self, text: str, fallback: dict) -> dict:
        """Parse JSON robuste : extrait le bloc JSON même si du texte l'entoure."""
        # Cherche le premier '{' et le dernier '}'
        start = text.find('{')
        end   = text.rfind('}')
        if start == -1 or end == -1:
            return fallback
        try:
            return _json.loads(text[start:end+1])
        except Exception:
            return fallback

    def _chunk_to_context(self, chunk: dict, with_score: str = "") -> str:
        art_id  = chunk.get('article', '?')
        ctx     = chunk.get('contexte', '')
        contenu = chunk.get('contenu', '')
        parts   = [f"══ الفصل {art_id} {'(' + ctx[:50] + ')' if ctx else ''} ══"]
        parts.append(contenu)
        return "\n".join(parts)

    def _chunk_to_display(self, chunk: dict,
                           score_info: str = "",
                           cot_reason: str = "") -> str:
        art_id  = chunk.get('article', '?')
        ctx     = chunk.get('contexte', '')
        contenu = chunk.get('contenu', '')
        lines   = [f"{'─'*55}", f"  الفصل {art_id}"]
        if ctx:
            lines.append(f"  [{ctx[:80]}]")
        if score_info:
            lines.append(f"  {score_info}")
        if cot_reason:
            lines.append(f"  → {cot_reason}")
        lines += ["", contenu, ""]
        return "\n".join(lines)

    def _lookup_direct(self, question: str) -> list:
        found, seen = [], set()
        for raw in self._ART_REF_RE.findall(question):
            num = raw.strip().replace('.', '-')
            cid = (self._num_idx.get(num)
                   or self._num_idx.get(str(_num_to_int(num))))
            if cid and cid not in seen:
                found.append(self.data[cid])
                seen.add(cid)
                self._log(f"Lookup direct → الفصل {self.data[cid].get('article', cid)}")
        return found

    # ── ÉTAPE 0 : Analyse de la question ─────────────────────────

    def _analyse_question(self, question: str) -> dict:
        self._log("ÉTAPE 0 — Analyse de la question")
        prompt   = _PROMPT_ANALYSE.format(question=question)
        raw      = self._llm_call(prompt, max_tokens=300)
        fallback = {
            "type": "factuelle", "domaines": [], "ambiguites": [],
            "mots_cles_arabes": [], "hors_loc": False
        }
        result = self._parse_json(raw, fallback)
        self._log(f"  Type    : {result.get('type')}")
        self._log(f"  Domaines: {result.get('domaines', [])}")
        if result.get('ambiguites'):
            self._log(f"  ⚠️  Ambiguïtés : {result['ambiguites']}")
        if result.get('hors_loc'):
            self._log("  ⚠️  Question potentiellement hors LOC")
        return result

    # ── ÉTAPE 1 : Retrieval étendu ────────────────────────────────

    def _retrieve_extended(self, question: str, analysis: dict,
                           top_k: int = 8) -> list:
        self._log("ÉTAPE 1 — Retrieval étendu")
        direct = self._lookup_direct(question)

        # Requête enrichie avec mots-clés extraits
        keywords  = analysis.get('mots_cles_arabes', [])
        enriched  = question
        if keywords:
            enriched = question + " " + " ".join(keywords[:4])

        semantic = self.retriever.retrieve_filtered(enriched, top_k=top_k)

        # Fusion sans doublons
        seen  = {c.get('article', '') for c in direct}
        final = list(direct)
        rrf_map = {}
        for r in semantic:
            art_id = r.chunk.get('article', r.chunk_id)
            rrf_map[art_id] = r.rrf_score
            if art_id not in seen:
                final.append(r.chunk)
                seen.add(art_id)

        self._log(f"  {len(final)} chunks candidats ({len(direct)} directs + "
                  f"{len(final)-len(direct)} RAG)")
        return final, rrf_map

    # ── ÉTAPE 2 : Re-ranking LLM ──────────────────────────────────

    def _rerank_chunks(self, question: str, analysis: dict,
                       candidates: list) -> tuple:
        self._log("ÉTAPE 2 — Re-ranking LLM des chunks")
        if not candidates:
            return [], []

        # Construction de la liste pour le prompt
        chunks_list = ""
        for chunk in candidates:
            art_id  = chunk.get('article', '?')
            contenu = chunk.get('contenu', '')[:150].replace('\n', ' ')
            ctx     = chunk.get('contexte', '')[:40]
            chunks_list += f"- الفصل {art_id} [{ctx}]: {contenu}…\n"

        prompt = _PROMPT_RERANKING.format(
            question=question,
            question_type=analysis.get('type', ''),
            domains=", ".join(analysis.get('domaines', [])),
            chunks_list=chunks_list,
        )
        raw      = self._llm_call(prompt, max_tokens=600)
        fallback = {"selected": [], "rejected": []}
        result   = self._parse_json(raw, fallback)

        selected_ids = {
            str(s.get('id', '')): s
            for s in result.get('selected', [])
        }
        rejected_ids = {
            str(r.get('id', ''))
            for r in result.get('rejected', [])
        }

        selected, rejected = [], []
        for chunk in candidates:
            art_id = str(chunk.get('article', ''))
            if art_id in selected_ids:
                info = selected_ids[art_id]
                chunk['_rerank_score']  = info.get('score', 0)
                chunk['_rerank_reason'] = info.get('raison', '')
                selected.append(chunk)
            elif art_id in rejected_ids:
                rejected.append(chunk)
            else:
                # Non classé → conserver avec score neutre
                chunk['_rerank_score']  = 5
                chunk['_rerank_reason'] = ''
                selected.append(chunk)

        # Trier par score décroissant
        selected.sort(key=lambda c: c.get('_rerank_score', 0), reverse=True)

        self._log(f"  Sélectionnés : {[c.get('article') for c in selected]}")
        self._log(f"  Rejetés      : {[c.get('article') for c in rejected]}")
        return selected, rejected

    # ── ÉTAPE 3 : Chain-of-Thought juridique ─────────────────────

    def _chain_of_thought(self, question: str, analysis: dict,
                          selected: list) -> dict:
        self._log("ÉTAPE 3 — Raisonnement juridique (CoT)")
        context = "\n\n".join(self._chunk_to_context(c) for c in selected[:6])
        prompt  = _PROMPT_COT.format(
            question=question,
            question_type=analysis.get('type', ''),
            context=context,
        )
        raw = self._llm_call(prompt, max_tokens=700)
        fallback = {
            "regle_juridique": "", "qualification": "",
            "conclusion": "", "reserves": [], "lacunes": [],
            "articles_cles": []
        }
        cot = self._parse_json(raw, fallback)
        self._log(f"  Conclusion CoT : {cot.get('conclusion', '')[:80]}…")
        if cot.get('lacunes'):
            self._log(f"  Lacunes détectées : {cot['lacunes']}")
        return cot

    # ── ÉTAPE 4a : Réponse client ─────────────────────────────────

    def _generate_client_answer(self, question: str, cot: dict,
                                selected: list) -> str:
        self._log("ÉTAPE 4a — Rédaction réponse client")
        arts_used = ", ".join(
            f"الفصل {c.get('article')}" for c in selected[:5]
        )
        prompt = _PROMPT_REPONSE_CLIENT.format(
            question=question,
            cot_json=_json.dumps(cot, ensure_ascii=False),
            articles_used=arts_used,
        )
        return self._llm_call(prompt, max_tokens=800, temperature=0.1)

    # ── ÉTAPE 4b : Analyse juriste ────────────────────────────────

    def _generate_juriste_answer(self, question: str, cot: dict,
                                 selected: list) -> str:
        self._log("ÉTAPE 4b — Rédaction analyse juriste")
        context = "\n\n".join(self._chunk_to_context(c) for c in selected[:6])
        prompt  = _PROMPT_REPONSE_JURISTE.format(
            question=question,
            cot_json=_json.dumps(cot, ensure_ascii=False),
            context=context,
        )
        return self._llm_call(prompt, max_tokens=1000, temperature=0.05)

    # ── ÉTAPE 5 : Auto-critique ───────────────────────────────────

    def _self_critique(self, question: str, answer: str,
                       selected: list) -> dict:
        self._log("ÉTAPE 5 — Auto-critique")
        arts_used = "\n".join(
            f"الفصل {c.get('article')}: {c.get('contenu','')[:100]}"
            for c in selected[:5]
        )
        prompt = _PROMPT_AUTOCRITIQUE.format(
            question=question,
            answer=answer[:600],
            articles_used=arts_used,
        )
        raw = self._llm_call(prompt, max_tokens=400)
        fallback = {
            "completude": "partielle",
            "interpretations_alternatives": [],
            "articles_manquants": [],
            "contradictions": [],
            "niveau_certitude": "moyenne",
            "avertissements": []
        }
        critique = self._parse_json(raw, fallback)
        self._log(f"  Complétude     : {critique.get('completude')}")
        self._log(f"  Certitude      : {critique.get('niveau_certitude')}")
        if critique.get('contradictions'):
            self._log(f"  ⚠️  Contradictions : {critique['contradictions']}")
        if critique.get('avertissements'):
            self._log(f"  ⚠️  Avertissements : {critique['avertissements']}")
        return critique

    # ── ÉTAPE 6 : Formatage sortie complète ──────────────────────

    def _format_output(self, question: str, analysis: dict, cot: dict,
                       selected: list, answer_client: str,
                       answer_juriste: str, critique: dict,
                       rrf_map: dict) -> str:

        confidence_map = {
            "haute": "🟢 HAUTE", "moyenne": "🟡 MOYENNE", "faible": "🔴 FAIBLE"
        }
        confidence = confidence_map.get(
            critique.get('niveau_certitude', 'moyenne'), "🟡 MOYENNE"
        )
        completude_map = {
            "complète": "✅", "partielle": "⚠️", "insuffisante": "❌"
        }
        completude = completude_map.get(critique.get('completude', ''), "⚠️")

        sep = "═" * 62

        # En-tête
        out = [
            f"\n{sep}",
            f"  CONSULTATION JURIDIQUE — LOC MAROCAIN",
            f"{sep}",
            f"  Question : {question}",
            f"  Type     : {analysis.get('type', '?')} | "
            f"Certitude : {confidence} | Complétude : {completude}",
        ]
        if analysis.get('domaines'):
            out.append(f"  Domaines : {', '.join(analysis['domaines'])}")
        if analysis.get('ambiguites'):
            out.append(f"  ⚠️  Ambiguïtés : {', '.join(analysis['ambiguites'])}")
        out.append(sep)

        # Réponse client
        out += [
            "",
            "  ┌─ RÉPONSE AU CLIENT ──────────────────────────────────────┐",
            "",
            answer_client,
            "",
            "  └──────────────────────────────────────────────────────────┘",
        ]

        # Analyse juriste
        out += [
            "",
            "  ┌─ ANALYSE JURIDIQUE TECHNIQUE ────────────────────────────┐",
            "",
            answer_juriste,
            "",
            "  └──────────────────────────────────────────────────────────┘",
        ]

        # Raisonnement intermédiaire (CoT synthèse)
        if cot.get('reserves') or cot.get('lacunes'):
            out += ["", "  ┌─ RÉSERVES & LACUNES ─────────────────────────────────────┐"]
            for r in cot.get('reserves', []):
                out.append(f"  ⚠️  {r}")
            for l in cot.get('lacunes', []):
                out.append(f"  🔍 Lacune : {l}")
            out.append("  └──────────────────────────────────────────────────────────┘")

        # Auto-critique
        if (critique.get('interpretations_alternatives')
                or critique.get('contradictions')
                or critique.get('avertissements')):
            out += ["", "  ┌─ AUTO-CRITIQUE ──────────────────────────────────────────┐"]
            for i in critique.get('interpretations_alternatives', []):
                out.append(f"  🔄 Alternative : {i}")
            for c in critique.get('contradictions', []):
                out.append(f"  ❗ Contradiction : {c}")
            for a in critique.get('avertissements', []):
                out.append(f"  ⚠️  {a}")
            out.append("  └──────────────────────────────────────────────────────────┘")

        # Textes officiels
        out += ["", f"  ┌─ TEXTES OFFICIELS UTILISÉS ({len(selected)}) ──────────────────┐"]
        for chunk in selected:
            art_id = chunk.get('article', '?')
            rrf    = rrf_map.get(art_id, 0)
            rerank = chunk.get('_rerank_score', '-')
            reason = chunk.get('_rerank_reason', '')
            score_info = f"[RRF={rrf:.4f} | Pertinence={rerank}/10]"
            out.append(self._chunk_to_display(chunk, score_info, reason))
        out.append("  └──────────────────────────────────────────────────────────┘")

        out.append(f"\n{sep}\n")
        return "\n".join(out)

    # ── Point d'entrée principal ──────────────────────────────────

    def ask(self, question: str, top_k: int = 8,
            mode: str = "complet") -> JuristAnalysis:
        """
        Paramètres :
          top_k : nombre de chunks RAG candidats (défaut 8)
          mode  : "complet" | "client" | "juriste"
                  complet  → réponse client + analyse technique + critique
                  client   → réponse claire uniquement (plus rapide)
                  juriste  → analyse technique uniquement
        """
        self.log = []
        print(f"\n{'═'*62}")
        print(f"  QUESTION : {question}")
        print(f"{'═'*62}")

        # 0. Analyse
        analysis = self._analyse_question(question)

        if analysis.get('hors_loc'):
            self._log("Question hors LOC détectée")
            answer = (
                "هذا السؤال لا يندرج ضمن نطاق قانون الالتزامات والعقود المغربي. "
                "يُنصح بالرجوع إلى النصوص القانونية المختصة (مدونة الشغل، "
                "قانون الأسرة، القانون التجاري...)."
            )
            print(answer)
            return JuristAnalysis(
                question=question, question_type="hors_loc",
                domains=[], ambiguities=[], chunks_retrieved=[],
                chunks_selected=[], chain_of_thought="", answer_client=answer,
                answer_juriste=answer, self_critique="", cited_articles=[],
                out_of_scope=True, confidence="faible", processing_log=self.log
            )

        # 1. Retrieval étendu
        candidates, rrf_map = self._retrieve_extended(question, analysis, top_k)

        # 2. Re-ranking
        selected, rejected = self._rerank_chunks(question, analysis, candidates)
        if not selected:
            selected = candidates[:5]   # fallback si le re-ranking échoue

        # 3. CoT
        cot = self._chain_of_thought(question, analysis, selected)

        # 4. Génération réponses
        answer_client  = ""
        answer_juriste = ""

        if mode in ("complet", "client"):
            answer_client  = self._generate_client_answer(question, cot, selected)
        if mode in ("complet", "juriste"):
            answer_juriste = self._generate_juriste_answer(question, cot, selected)

        main_answer = answer_juriste or answer_client

        # 5. Auto-critique
        critique = self._self_critique(question, main_answer, selected)

        # 6. Formatage
        output = self._format_output(
            question, analysis, cot, selected,
            answer_client, answer_juriste, critique, rrf_map
        )
        print(output)

        return JuristAnalysis(
            question=question,
            question_type=analysis.get('type', ''),
            domains=analysis.get('domaines', []),
            ambiguities=analysis.get('ambiguites', []),
            chunks_retrieved=candidates,
            chunks_selected=selected,
            chain_of_thought=_json.dumps(cot, ensure_ascii=False),
            answer_client=answer_client,
            answer_juriste=answer_juriste,
            self_critique=_json.dumps(critique, ensure_ascii=False),
            cited_articles=cot.get('articles_cles', []),
            out_of_scope=False,
            confidence=critique.get('niveau_certitude', 'moyenne'),
            processing_log=self.log,
        )

# ─────────────────────────────────────────────────────────────────
# INSTANCIATION
# ─────────────────────────────────────────────────────────────────

jurist_agent = LawJuristAgent(
    retriever=retriever,
    llm=llm,
    chunks_dict=chunks,
    chunk_ids=chunk_ids,
    verbose=True,
)
print("  Agent juriste prêt.\n")

# ─────────────────────────────────────────────────────────────────
# TESTS
# ─────────────────────────────────────────────────────────────────

# Test 1 — question interprétative (l'exemple problématique d'origine)
r1 = jurist_agent.ask(
    "ما هي مدة عقد العمل غير المحدد المدة وهل يمكن إنهاؤه؟",
    top_k=8, mode="complet"
)

# Test 2 — question factuelle directe avec numéro de فصل
# r2 = jurist_agent.ask("ما الذي ينص عليه الفصل 230 من قانون الالتزامات؟")

# Test 3 — question comparative
# r3 = jurist_agent.ask(
#     "ما الفرق بين الفسخ والإبطال في القانون المغربي؟",
#     mode="juriste"
# )

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Interface interactive
# ══════════════════════════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

CSS = """<style>
  .loc-title  { font-size:1.4em; font-weight:bold; color:#1a237e;
                text-align:center; margin-bottom:8px; direction:rtl; }
  .loc-sub    { font-size:0.9em; color:#555; text-align:center; margin-bottom:16px; }
  .loc-art    { background:#fff8e1; border-left:4px solid #f9a825;
                border-radius:4px; padding:12px; margin-top:10px; direction:rtl; }
  .loc-num    { color:#b71c1c; font-weight:bold; font-size:1.05em; }
  .loc-spin   { color:#1a237e; font-style:italic; }
</style>"""

title_html = widgets.HTML(CSS + """
<div class="loc-title">المستشار القانوني — قانون الالتزامات والعقود المغربي</div>
<div class="loc-sub">اكتب سؤالك القانوني وسيعرض النظام الفصل الكامل والنص الرسمي</div>
""")

question_input = widgets.Textarea(
    placeholder="مثال: ما هي شروط صحة العقد؟\nمثال: متى يجوز فسخ العقد؟",
    layout=widgets.Layout(width="100%", height="90px"),
)
top_k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1, description="عدد الفصول:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="50%"),
)
submit_btn = widgets.Button(description="إرسال السؤال", button_style="primary",
                            layout=widgets.Layout(width="200px", height="40px"))
clear_btn  = widgets.Button(description="مسح", button_style="warning",
                            layout=widgets.Layout(width="100px", height="40px"))
output_area = widgets.Output(layout=widgets.Layout(width="100%", border="1px solid #ddd",
                             border_radius="8px", padding="8px", min_height="100px"))


def on_submit(b):
    question = question_input.value.strip()
    if not question:
        with output_area:
            clear_output()
            display(HTML('<p style="color:red;text-align:right">الرجاء كتابة سؤال</p>'))
        return

    with output_area:
        clear_output()
        display(HTML('<p class="loc-spin">جارٍ التحليل والبحث في الفصول القانونية…</p>'))

    try:
        result = agent.ask(question, top_k=top_k_slider.value)
    except Exception as exc:
        with output_area:
            clear_output()
            display(HTML(f'<p style="color:red">خطأ : {exc}</p>'))
        return

    def _esc(s):
        return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    # Séparer réponse LLM et textes officiels
    sep = "TEXTES OFFICIELS DES ARTICLES UTILISES"
    parts = result.split(sep)
    llm_part  = parts[0].strip() if parts else result
    arts_part = parts[1].strip() if len(parts) > 1 else ""

    def _clean_sep(text):
        lines = [l for l in text.splitlines() if not re.match(r"^[=\-]{3,}$", l.strip())]
        return "\n".join(lines).strip()

    llm_clean  = _clean_sep(llm_part.replace("REPONSE DE L'AGENT JURIDIQUE", "").strip())
    arts_clean = _clean_sep(arts_part)

    html = [CSS, '<div style="direction:rtl;font-family:\'Amiri\',Arial,serif;">',
            '<div style="background:#e8f5e9;border-radius:8px;padding:14px;margin-bottom:12px;">',
            '<div style="color:#1b5e20;font-weight:bold;font-size:1.1em;margin-bottom:8px;">الإجابة القانونية</div>',
            f'<div style="white-space:pre-wrap;font-size:0.95em;">{_esc(llm_clean)}</div>',
            '</div>',
            '<div style="color:#4a148c;font-weight:bold;font-size:1em;margin:12px 0 8px;">النصوص الرسمية للفصول المستخدمة</div>']

    for block in re.split(r"─{3,}", arts_clean):
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        html += ['<div class="loc-art">',
                 f'<div class="loc-num">{_esc(lines[0].strip() if lines else "")}</div>',
                 f'<div style="margin-top:8px;white-space:pre-wrap;font-size:0.92em;">{_esc(chr(10).join(lines[1:]).strip())}</div>',
                 '</div>']

    html.append('</div>')
    with output_area:
        clear_output()
        display(HTML("".join(html)))


def on_clear(b):
    question_input.value = ""
    with output_area:
        clear_output()


submit_btn.on_click(on_submit)
clear_btn.on_click(on_clear)

display(widgets.VBox(
    [title_html, question_input,
     widgets.HBox([submit_btn, clear_btn, top_k_slider],
                  layout=widgets.Layout(gap="12px", align_items="center", margin="8px 0")),
     output_area],
    layout=widgets.Layout(width="100%", padding="12px"),
))


Question : ما هي المواد التي تنظم عقد العمل في مدونة الشغل؟

Articles injectés dans le LLM (5) :
   الفصل     3-65 [RAG] [RRF=0.0320] — 1214 chars
   الفصل   2.1197 [RAG] [RRF=0.0315] — 1199 chars
   الفصل      443 [RAG] [RRF=0.0307] — 199383 chars
   الفصل     7-65 [RAG] [RRF=0.0303] — 12172 chars
   الفصل    106-1 [RAG] [RRF=0.0298] — 73123 chars

Génération de la réponse…

Question : ما هي المواد التي تنظم عقد العمل في مدونة الشغل؟

Articles injectés dans le LLM (5) :
   الفصل     3-65 [RAG] [RRF=0.0320] — 1214 chars
   الفصل   2.1197 [RAG] [RRF=0.0315] — 1199 chars
   الفصل      443 [RAG] [RRF=0.0307] — 199383 chars
   الفصل     7-65 [RAG] [RRF=0.0303] — 12172 chars
   الفصل    106-1 [RAG] [RRF=0.0298] — 73123 chars

Génération de la réponse…
